# Riyadh dust: revision notebook (Atmosphere, manuscript atmosphere-4564308)

This notebook supports the major revision. It **does not rerun HYSPLIT, the METAR classification or the ERA5
download**; it reads the files your original notebook (`Riyadh_Dust.ipynb`, left unchanged) already wrote to
Drive, and writes everything new to `07_Revision/` inside the project folder.

## What you need to run

| Step | Cells | Needed? | Time | What it produces |
|---|---|---|---|---|
| **A** | **1.1 and all cells of Section 1** (module files) | **Yes** | seconds | Writes the analysis code to `/content/rev_pkg` |
| **B** | **2.1 Export trajectory statistics** | **Once** (already done if you ran the R0 cell; it then skips itself) | ~10 min | `07_Revision/REV_traj_cellcounts.npz` (re-reads the existing `tdump` files) |
| **C** | **7.1 Trajectory re-analysis** | **Yes** | ~2 min | Tables 4, 5 and S1, Figures 7-8 (Reviewer 3, major comments 1, 2, 4) |
| D | 3.1 Compact METAR export | Optional | seconds | Adds a third CWT intensity (visibility at arrival); not used in the paper |
| E | 4.1, 5.1, 6.1 | Optional | ~2 min | Recreates Table 1, Tables 2-3, Figures 1-6 (already in the revised manuscript) |
| F | 8.1-8.2 Upwind ERA5 | Optional | 10-40 min (CDS queue) | First look at upwind soil moisture, rain and wind (not used in the manuscript) |

**Minimum: run 1.1 → Section 1 → 2.1 → 7.1** (Runtime → Run before, with 7.1 selected, does exactly this; 2.1
returns immediately when the export already exists). Sections 4–6 and 8 are optional.

Outputs go to `07_Revision/outputs/` (CSV), `07_Revision/outputs/trajectory/` (trajectory results) and
`07_Revision/figures/` (PNG, 300 dpi).

## Map from reviewer comments to cells

| Comment | Where |
|---|---|
| R1-1 figure annotations obscure results | 6.1 (Figs 1-6), 7.1 (Figs 7-8): no text boxes inside axes |
| R1-4 / R3-1 "suspension-dominated" label, exclusive classes, imperfect control | 4.1 (counts), 7.1 (`Lonly`, `Sonly` fields, nulls) |
| R3-2 exact estimator, base rate, null, coverage | 7.1: `rev_traj_run.py` (Eqs. 1-4 of the revised text), `Table4_regions.csv` |
| R3-3 units and reconciliation of Table 5 | 7.1: `Table5_counts_by_wind_quintile.csv`, `summary.json` |
| R3-4 class-specific CWT, alternative intensity | 7.1: `CWT_daymax`, `CWT_hours` (Figure 7, `TableS1_*.csv`) |
| R3-5 calibration, benchmarks, prevalence | 5.1 (`calib_yearly.csv`, `calib_skill.csv`), 4.1 (Table 1) |
| R3 minor: 41 unassigned reports, visibility ceiling, Table 1 uncertainty | 4.1 (`hourly_checks.csv`, `table1_changes.csv`) |


## 1. Setup
Run **1.1**, then every cell of this section once per session. The module cells only write `.py` files; edit them here if you want to change an analysis, then rerun the cell and the step that uses it.

In [ ]:
# 1.1  Mount Drive and point to the project folder
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = "/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution"   # <- edit if your folder differs
os.environ["RIYADH_PROJECT"] = PROJECT
PKG = "/content/rev_pkg"
os.makedirs(PKG, exist_ok=True)
if PKG not in sys.path:
    sys.path.insert(0, PKG)

%load_ext autoreload
%autoreload 2

import numpy, pandas, scipy, sklearn, matplotlib
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| scipy", scipy.__version__,
      "| scikit-learn", sklearn.__version__, "| matplotlib", matplotlib.__version__)
print("Project folder found:", os.path.isdir(PROJECT))

**`paths.py`**: Folder layout. Edit only if your Drive folders differ from the original notebook.

In [ ]:
%%writefile /content/rev_pkg/paths.py
"""Central paths for the revision analyses (Colab defaults; override with env vars)."""
import os
from pathlib import Path

PROJECT = Path(os.environ.get("RIYADH_PROJECT", "/content/drive/MyDrive/Research/Riyadh_Dust_Source_Attribution"))
CLASS_CSV = Path(os.environ.get("CLASS_CSV", PROJECT / "04_Analysis" / "CLASS_daily_events.csv"))
METAR_HOURLY = Path(os.environ.get("METAR_HOURLY", PROJECT / "03_Observations" / "METAR" / "processed" / "METAR_hourly_riyadh.csv"))
HOMOG_CSV = Path(os.environ.get("HOMOG_CSV", PROJECT / "03_Observations" / "METAR" / "processed" / "HOMOGENEITY_annual.csv"))
REV = Path(os.environ.get("REV_DIR", PROJECT / "07_Revision"))
OUT = REV / "outputs"
FIGS = REV / "figures"
NPZ = REV / "REV_traj_cellcounts.npz"
METAR_COMPACT = REV / "REV_metar_compact.csv.gz"
UPWIND_CSV = REV / "REV_upwind_monthly.csv"
NE_DIR = REV / "naturalearth"
for p in (REV, OUT, FIGS, NE_DIR):
    p.mkdir(parents=True, exist_ok=True)


**`figstyle.py`**: Shared figure style (fonts, class colours).

In [ ]:
%%writefile /content/rev_pkg/figstyle.py
"""Shared figure style for the revision figures (Atmosphere, MDPI)."""
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Liberation Serif"],
    "font.size": 9,
    "axes.titlesize": 9.5,
    "axes.labelsize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": ":",
    "legend.frameon": False,
})

C_SUS = "#2c7fb8"      # "suspension-dominated" (DU/SA)
C_LOC = "#e67e22"      # locally raised (BLDU/BLSA)
C_BOTH = "#8e6bb3"     # days meeting both criteria
C_NEI = "0.70"         # dust days meeting neither class criterion
C_ALL = "0.25"
SEG = [("Baseline", 2016, 2022), ("Minimum", 2023, 2024), ("Recovery", 2025, 2025)]
SEG_COL = {"Baseline": "0.55", "Minimum": "#c0392b", "Recovery": "#e67e22"}

LAB_SUS = "“Suspension-dominated” (DU/SA)"
LAB_LOC = "Locally raised (BLDU/BLSA)"


def shade_min(ax):
    ax.axvspan(2022.5, 2024.5, color="#c0392b", alpha=0.07, zorder=0, lw=0)


**`rev_maps.py`**: Base map: Natural Earth coastlines (downloaded once), receptor, 300 km circle, source boxes.

In [ ]:
%%writefile /content/rev_pkg/rev_maps.py
"""Base-map helpers (Natural Earth coastlines and borders, receptor, exclusion circle, regions)."""
import json
import urllib.request
from pathlib import Path

import numpy as np

RECEPTOR = (24.9578, 46.6989)
ERA5_PT = (24.75, 46.75)
EXCL_KM = 300.0
REGIONS = {
    "Mesopotamia / Fertile Crescent": (30.0, 37.0, 38.0, 49.0),
    "Syrian Desert": (31.0, 35.5, 36.0, 41.0),
    "An Nafud": (27.0, 30.5, 38.0, 43.0),
    "Ad-Dahna corridor": (21.0, 28.0, 44.0, 48.5),
    "Rub al Khali": (16.0, 23.0, 45.0, 56.0),
    "Eastern Arabia / Gulf": (24.0, 30.0, 47.0, 53.0),
    "Iranian plateau / Sistan": (28.0, 35.0, 54.0, 63.0),
    "Northeast Africa": (15.0, 32.0, 25.0, 37.0),
}
NE_URL = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/{}.geojson"
_CACHE = {}


def _lines(name, ne_dir):
    if name in _CACHE:
        return _CACHE[name]
    p = Path(ne_dir) / f"{name}.geojson"
    if not p.exists():
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(NE_URL.format(name), p)
        except Exception as e:  # offline: draw without coastlines
            print(f"  (coastlines unavailable: {e})")
            _CACHE[name] = []
            return []
    gj = json.loads(p.read_text())
    out = []
    for f in gj["features"]:
        g = f["geometry"]
        parts = [g["coordinates"]] if g["type"] == "LineString" else g["coordinates"]
        for part in parts:
            a = np.asarray(part)
            if ((a[:, 0] > 15) & (a[:, 0] < 70) & (a[:, 1] > 5) & (a[:, 1] < 50)).any():
                out.append(a)
    _CACHE[name] = out
    return out


def basemap(ax, ne_dir, lon=(25, 65), lat=(10, 45), labels=True):
    for a in _lines("ne_50m_coastline", ne_dir):
        ax.plot(a[:, 0], a[:, 1], color="0.25", lw=0.55, zorder=4)
    for a in _lines("ne_50m_admin_0_boundary_lines_land", ne_dir):
        ax.plot(a[:, 0], a[:, 1], color="0.55", lw=0.35, zorder=4)
    ax.set_xlim(*lon)
    ax.set_ylim(*lat)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    if labels:
        ax.set_xlabel("Longitude (°E)")
        ax.set_ylabel("Latitude (°N)")


def excl_circle(ax, color="k", lw=1.1):
    t = np.linspace(0, 2 * np.pi, 200)
    r_lat = EXCL_KM / 111.2
    r_lon = r_lat / np.cos(np.radians(RECEPTOR[0]))
    ax.plot(RECEPTOR[1] + r_lon * np.cos(t), RECEPTOR[0] + r_lat * np.sin(t), ls=":", color=color, lw=lw, zorder=6)


def receptor(ax, ms=11):
    ax.plot(RECEPTOR[1], RECEPTOR[0], marker="*", ms=ms, color="white", mec="k", mew=0.8, zorder=7, ls="none")


def region_boxes(ax, color="k", lw=0.7, numbers=False):
    from matplotlib.patches import Rectangle
    for i, (name, (la0, la1, lo0, lo1)) in enumerate(REGIONS.items(), 1):
        ax.add_patch(Rectangle((lo0, la0), lo1 - lo0, la1 - la0, fill=False, edgecolor=color, lw=lw,
                               ls="--", zorder=5))
        if numbers:
            ax.text(lo0 + 0.3, la1 - 0.3, str(i), fontsize=6.5, va="top", ha="left", zorder=6)


**`rev_export.py`**: Exports compact inputs from files already on Drive (no HYSPLIT rerun).

In [ ]:
%%writefile /content/rev_pkg/rev_export.py
"""Export compact inputs for the revision from files that already exist on Drive.

export_traj()  reads the HYSPLIT tdump files (or the cached endpoint parquet) and writes
               07_Revision/REV_traj_cellcounts.npz: per-trajectory endpoint counts on the
               1-degree grid after the 3 h and 300 km filters. HYSPLIT is NOT rerun.
export_metar() writes 07_Revision/REV_metar_compact.csv.gz (four columns of the hourly file).
"""
import numpy as np
import pandas as pd

import paths as P

TRAJ_DIR = P.PROJECT / "05_HYSPLIT" / "trajectories"
PARQ = P.PROJECT / "06_SourceAttribution" / "PSCF_endpoints.parquet"
R_LAT, R_LON = 24.9578, 46.6989
LAT_EDGES = np.arange(10.0, 46.0, 1.0)   # 10-45 N, 1 deg (as the original source-attribution cell)
LON_EDGES = np.arange(25.0, 66.0, 1.0)   # 25-65 E


def parse_tdump(path):
    """Same parser as the original notebook: age, lat, lon, height."""
    rows = []
    with open(path) as fh:
        for line in fh:
            p = line.split()
            if len(p) < 12:
                continue
            try:
                v = [float(x) for x in p]
            except ValueError:
                continue
            rows.append((v[8], v[9], v[10], v[11]))
    return np.array(rows, dtype=np.float32) if rows else None


def export_traj(force=False):
    out = P.NPZ
    if out.exists() and not force:
        print(f"{out.name} already exists ({out.stat().st_size / 1e6:.1f} MB); use force=True to rebuild.")
        return out
    if PARQ.exists():
        ep = pd.read_parquet(PARQ, columns=["tag", "age", "lat", "lon", "height"])
        print(f"Loaded cached endpoints: {len(ep):,}")
    else:
        files = sorted(TRAJ_DIR.glob("tdump_*"))
        print(f"Parsing {len(files):,} trajectory files (no HYSPLIT rerun) ...")
        frames = []
        for i, f in enumerate(files, 1):
            a = parse_tdump(f)
            if a is not None:
                frames.append(pd.DataFrame({"tag": f.name.replace("tdump_", ""), "age": a[:, 0],
                                            "lat": a[:, 1], "lon": a[:, 2], "height": a[:, 3]}))
            if i % 2000 == 0:
                print(f"  {i:,}/{len(files):,}")
        ep = pd.concat(frames, ignore_index=True)
    tags = np.array(sorted(ep["tag"].unique()))
    ep["ti"] = pd.Series(np.arange(len(tags)), index=tags).loc[ep["tag"].values].values
    lat = ep["lat"].values.astype(np.float64)
    lon = ep["lon"].values.astype(np.float64)
    la0, lo0 = np.radians(R_LAT), np.radians(R_LON)
    a = (np.sin((np.radians(lat) - la0) / 2) ** 2
         + np.cos(la0) * np.cos(np.radians(lat)) * np.sin((np.radians(lon) - lo0) / 2) ** 2)
    dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    age_ok = np.abs(ep["age"].values) >= 3
    dist_ok = dist >= 300.0
    i = np.searchsorted(LAT_EDGES, lat, side="right") - 1          # numpy.histogram2d convention
    j = np.searchsorted(LON_EDGES, lon, side="right") - 1
    i[lat == LAT_EDGES[-1]] = len(LAT_EDGES) - 2
    j[lon == LON_EDGES[-1]] = len(LON_EDGES) - 2
    in_grid = (i >= 0) & (i < len(LAT_EDGES) - 1) & (j >= 0) & (j < len(LON_EDGES) - 1)
    sel = age_ok & dist_ok & in_grid
    cell = i * (len(LON_EDGES) - 1) + j
    trip = (pd.DataFrame({"ti": ep["ti"].values[sel], "cell": cell[sel]})
            .groupby(["ti", "cell"]).size().reset_index(name="n"))
    per = pd.DataFrame({"ti": ep["ti"].values, "all": 1, "age_ok": age_ok, "age_dist_ok": age_ok & dist_ok,
                        "kept": sel}).groupby("ti")[["all", "age_ok", "age_dist_ok", "kept"]].sum()
    per = per.reindex(np.arange(len(tags)), fill_value=0)
    np.savez_compressed(out, tags=tags.astype("U11"), ti=trip["ti"].values.astype(np.int32),
                        cell=trip["cell"].values.astype(np.int16), n=trip["n"].values.astype(np.int16),
                        n_all=per["all"].values.astype(np.int16), n_age_ok=per["age_ok"].values.astype(np.int16),
                        n_age_dist_ok=per["age_dist_ok"].values.astype(np.int16),
                        n_kept=per["kept"].values.astype(np.int16), lat_edges=LAT_EDGES, lon_edges=LON_EDGES)
    print(f"Trajectories          : {len(tags):,}")
    print(f"Endpoints (all)       : {len(ep):,}")
    print(f"After age + 300 km    : {int((age_ok & dist_ok).sum()):,}")
    print(f"...and inside the grid: {int(sel.sum()):,}")
    print(f"Saved -> {out}  ({out.stat().st_size / 1e6:.1f} MB)")
    return out


def export_metar():
    h = pd.read_csv(P.METAR_HOURLY, usecols=["valid", "vsby", "vis_km", "wxcodes"], low_memory=False)
    h.to_csv(P.METAR_COMPACT, index=False, compression="gzip")
    print(f"{len(h):,} METAR rows -> {P.METAR_COMPACT}")
    return P.METAR_COMPACT


**`rev_stats.py`**: Catalogue statistics: overlap and exclusive classes, Table 1 with bootstrap intervals, hourly checks.

In [ ]:
%%writefile /content/rev_pkg/rev_stats.py
"""Catalogue statistics for the revision: overlap/exclusive classes, Table 1 regime
statistics with bootstrap intervals, rank test, yearly prevalence (for reference; not tabulated in the paper), and the
hourly checks (41 unassigned reports, visibility ceiling)."""
from math import comb

import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint

import paths as P

HOM_FALLBACK = pd.DataFrame({
    "year": range(2016, 2026),
    "dust_rate": [0.0465, 0.0704, 0.0843, 0.0613, 0.0511, 0.0436, 0.0910, 0.0084, 0.0096, 0.0282],
    "nondust_wx_rate": [0.0197, 0.0168, 0.0235, 0.0251, 0.0147, 0.0105, 0.0192, 0.0333, 0.0132, 0.0215],
    "vis_low_hours": [383, 519, 634, 440, 359, 346, 718, 178, 57, 262],
    "vis_storm_hours": [50, 53, 184, 56, 45, 57, 84, 19, 6, 46],
    "frac_lowvis_coded_dust": [0.7467, 0.8748, 0.8596, 0.7909, 0.8189, 0.8179, 0.8231, 0.2360, 0.8246, 0.6603],
})


def load_class():
    d = pd.read_csv(P.CLASS_CSV, parse_dates=["date"])
    d = d[d.date.dt.year <= 2025].copy()
    d["year"], d["month"] = d.date.dt.year, d.date.dt.month
    d["L"], d["S"], d["D"] = d.local_day == 1, d.transport_day == 1, d.dust_day == 1
    d["both"], d["Lonly"], d["Sonly"] = d.L & d.S, d.L & ~d.S, d.S & ~d.L
    d["neither"] = d.D & ~d.L & ~d.S
    return d


def load_hom():
    if P.HOMOG_CSV.exists():
        h = pd.read_csv(P.HOMOG_CSV)
        return h[h.year.between(2016, 2025)].reset_index(drop=True)
    return HOM_FALLBACK.copy()


def class_counts(d):
    out = {}
    for per, (a, b) in {"2016-2025": (2016, 2025), "2016-2022": (2016, 2022), "2023-2025": (2023, 2025)}.items():
        s = d[d.year.between(a, b)]
        out[per] = {k: int(s[k].sum()) for k in ["D", "L", "S", "both", "Lonly", "Sonly", "neither"]}
    nl = d[d.D & ~d.L]
    out["dust_days_not_local"] = int(len(nl))
    out["of_which_1_2_local_hours"] = int((nl.local_hours >= 1).sum())
    return out


def table1(d, B=20000, seed=42):
    h = load_hom().rename(columns={"nondust_wx_rate": "other_rate", "vis_low_hours": "vis5", "vis_storm_hours": "vis1"})
    cnt = d.groupby("year").agg(days=("D", "sum"), loc=("L", "sum"), sus=("S", "sum")).reset_index()
    t = h[["year", "dust_rate", "other_rate", "vis5", "vis1"]].merge(cnt, on="year")
    t.to_csv(P.OUT / "table1_annual.csv", index=False)
    rng = np.random.default_rng(seed)
    base, mn, rc = t[t.year <= 2022], t[t.year.isin([2023, 2024])], t[t.year == 2025]
    rows = []
    for c in ["dust_rate", "other_rate", "vis5", "vis1", "days", "loc", "sus"]:
        bv = base[c].values.astype(float)
        boots = rng.choice(bv, (B, len(bv)), replace=True).mean(1)
        for name, sub in [("Min", mn), ("Rec", rc)]:
            v = sub[c].mean()
            ci = np.percentile(100 * (v - boots) / boots, [2.5, 97.5])
            rows.append(dict(metric=c, regime=name, base_mean=bv.mean(), base_min=bv.min(), base_max=bv.max(),
                             value=v, pct=100 * (v - bv.mean()) / bv.mean(), ci_lo=ci[0], ci_hi=ci[1]))
    ch = pd.DataFrame(rows)
    ch.to_csv(P.OUT / "table1_changes.csv", index=False)
    ranks = {c: t.sort_values(c).year.tolist()[:3] for c in ["dust_rate", "vis5", "vis1", "days", "loc", "sus"]}
    return t, ch, ranks, 1 / comb(10, 3)


def prevalence(d):
    rows = []
    for y, g in d.groupby("year"):
        r = {"year": y, "n_days": len(g)}
        for c in ["D", "L", "S", "both", "Lonly", "Sonly"]:
            k = int(g[c].sum())
            lo, hi = proportion_confint(k, len(g), method="wilson")
            r.update({c: k, c + "_pct": 100 * k / len(g), c + "_lo": 100 * lo, c + "_hi": 100 * hi})
        rows.append(r)
    p = pd.DataFrame(rows)
    p.to_csv(P.OUT / "prevalence_by_year.csv", index=False)
    return p


def hourly_checks():
    """Reconcile hourly class counts and check the visibility ceiling (needs the METAR file)."""
    src = P.METAR_COMPACT if P.METAR_COMPACT.exists() else P.METAR_HOURLY
    if not src.exists():
        return None
    h = pd.read_csv(src, low_memory=False)
    tok = h["wxcodes"].fillna("").astype(str).str.split().apply(set)
    dust = {"DU", "BLDU", "DS", "SA", "BLSA", "SS", "PO", "VCBLDU", "VCBLSA", "VCDS", "VCSS", "+DS", "+SS", "-DS", "-SS"}
    is_dust = tok.apply(lambda t: bool(dust & t))
    local = tok.apply(lambda t: bool({"BLDU", "BLSA"} & t))
    susp = tok.apply(lambda t: bool({"DU", "SA"} & t) and not bool({"BLDU", "BLSA"} & t))
    storm = tok.apply(lambda t: bool({"DS", "SS"} & t) and not bool({"BLDU", "BLSA", "DU", "SA"} & t))
    po = tok.apply(lambda t: "PO" in t and not bool({"BLDU", "BLSA", "DU", "SA", "DS", "SS"} & t))
    other = is_dust & ~local & ~susp & ~storm & ~po
    res = {"dust_obs": int(is_dust.sum()), "local": int(local.sum()), "suspension": int(susp.sum()),
           "DS_SS_only": int(storm.sum()), "PO_only": int(po.sum()), "other_dust_codes": int(other.sum()),
           "vis_km_max": float(h["vis_km"].max()), "n_vis_above_9.999": int((h["vis_km"] > 9.9995).sum()),
           "vsby_max_miles": float(pd.to_numeric(h["vsby"], errors="coerce").max())}
    pd.Series(res).to_csv(P.OUT / "hourly_checks.csv")
    return res


if __name__ == "__main__":
    d = load_class()
    print(class_counts(d))
    t, ch, ranks, p = table1(d)
    print(ch.round(2).to_string(index=False)); print(ranks, p)
    print(prevalence(d).round(1).to_string(index=False))
    print(hourly_checks())


**`rev_calib.py`**: Calibrated expectation and seasonal-climatology benchmark (Tables 2-3, Figure 6). Runs when executed.

In [ ]:
%%writefile /content/rev_pkg/rev_calib.py
import numpy as np, pandas as pd, json
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
import paths as P
IN=P.CLASS_CSV
BASE=["wind_speed_max","swvl1_mean","precip_sum","precip_7d","precip_30d","vpd_max","blh_max","t2m_max","ssrd_mean"]
df=pd.read_csv(IN,low_memory=False); df["date"]=pd.to_datetime(df.date)
df["year"]=df.date.dt.year; df["month"]=df.date.dt.month; df["doy"]=df.date.dt.dayofyear
df=df[df.year<=2025].copy()
th=np.radians(df.wind_dir_vec); df["wind_dir_sin"]=np.sin(th); df["wind_dir_cos"]=np.cos(th)
df["wind_projection"]=df.wind_speed_max*np.cos(th-np.radians(320.0))
df["doy_sin"]=np.sin(2*np.pi*df.doy/365.25); df["doy_cos"]=np.cos(2*np.pi*df.doy/365.25)
F=BASE+["wind_dir_sin","wind_dir_cos","wind_projection","doy_sin","doy_cos"]
def rf(): return RandomForestClassifier(n_estimators=400,min_samples_leaf=5,class_weight="balanced",random_state=0,n_jobs=-1)
def lg(x): x=np.clip(x,1e-4,1-1e-4); return np.log(x/(1-x))
rng=np.random.default_rng(7)
out_rows=[]; skill=[]; store={}
for target,label in [("local_day","Locally lifted"),("transport_day","Suspension-dominated"),("dust_day","All dust")]:
    d=df.dropna(subset=[target]+F).copy()
    tr=d[d.year<=2022].copy(); te=d[d.year>=2023].copy()
    m=rf().fit(tr[F].values,tr[target].values)
    te["s"]=m.predict_proba(te[F].values)[:,1]
    tr["s"]=np.nan
    for y in range(2016,2023):
        a=tr[tr.year!=y]; b=tr.year==y
        mm=rf().fit(a[F].values,a[target].values)
        tr.loc[b,"s"]=mm.predict_proba(tr.loc[b,F].values)[:,1]
    # Platt calibration on OOF scores
    pl=LogisticRegression(C=1e6,max_iter=1000).fit(lg(tr.s.values)[:,None],tr[target].values)
    tr["p"]=pl.predict_proba(lg(tr.s.values)[:,None])[:,1]; te["p"]=pl.predict_proba(lg(te.s.values)[:,None])[:,1]
    # climatologies from training years
    pbar=tr[target].mean(); mon=tr.groupby("month")[target].mean()
    for x in (tr,te): x["clim"]=pbar; x["sclim"]=x.month.map(mon)
    # bootstrap of calibration map over training years (block bootstrap by year)
    yrs=np.arange(2016,2023); Bc=500; cal_te=[]; cal_tr=[]
    for k in range(Bc):
        ys=rng.choice(yrs,len(yrs),replace=True)
        bb=pd.concat([tr[tr.year==y] for y in ys])
        if bb[target].sum()<5: continue
        plb=LogisticRegression(C=1e6,max_iter=1000).fit(lg(bb.s.values)[:,None],bb[target].values)
        cal_te.append(plb.predict_proba(lg(te.s.values)[:,None])[:,1]); cal_tr.append(plb.predict_proba(lg(tr.s.values)[:,None])[:,1])
    cal_te=np.array(cal_te); cal_tr=np.array(cal_tr)
    allx=pd.concat([tr,te]); allcal=np.concatenate([cal_tr,cal_te],axis=1)
    # dispersion factor from baseline years (OOF)
    g=tr.groupby("year").agg(O=(target,"sum"),E=("p","sum"),V=("p",lambda p:(p*(1-p)).sum()))
    phi=max(1.0,((g.O-g.E)**2).sum()/g.V.sum())
    for y in range(2016,2026):
        sel=(allx.year==y).values; x=allx[sel]
        O=x[target].sum(); E=x.p.sum(); Ec=x.clim.sum(); Es=x.sclim.sum()
        # PI: calibration uncertainty (bootstrap) + overdispersed binomial sampling
        Eb=allcal[:,sel].sum(1); Vb=(allcal[:,sel]*(1-allcal[:,sel])).sum(1)
        sims=Eb+np.sqrt(phi*Vb)*rng.standard_normal(len(Eb))
        sims=np.concatenate([Eb[:,None]+np.sqrt(phi*Vb)[:,None]*rng.standard_normal((len(Eb),40))]).ravel()
        lo,hi=np.percentile(np.clip(sims,0,None),[2.5,97.5])
        out_rows.append(dict(cls=label,year=y,period="train (OOF)" if y<=2022 else "test",n=len(x),O=int(O),E_model=E,PI_lo=lo,PI_hi=hi,E_clim=Ec,E_sclim=Es,O_over_E=O/E if E>0 else np.nan,met_index=E/Es,phi=phi))
    # skill on test period
    yt=te[target].values
    skill.append(dict(cls=label,AUC_model=roc_auc_score(yt,te.s),AUC_sclim=roc_auc_score(yt,te.sclim),
        Brier_model=brier_score_loss(yt,te.p),Brier_sclim=brier_score_loss(yt,te.sclim),Brier_clim=brier_score_loss(yt,te.clim),
        AUC_oof=roc_auc_score(tr[target],tr.s),AUC_sclim_train=roc_auc_score(tr[target],tr.sclim),
        Brier_oof=brier_score_loss(tr[target],tr.p),Brier_sclim_train=brier_score_loss(tr[target],tr.sclim),
        raw_ratio_train=tr.s.mean()/tr[target].mean(), raw_ratio_test=te.s.mean()/te[target].mean(), platt_a=pl.intercept_[0], platt_b=pl.coef_[0][0]))
    store[target]=dict(tr=tr[["date","year","month",target,"s","p","clim","sclim"]],te=te[["date","year","month",target,"s","p","clim","sclim"]])
    tr[["date","year","month",target,"s","p","sclim"]].assign(split="train_oof").to_csv(P.OUT/f"calib_{target}_train.csv",index=False)
    te[["date","year","month",target,"s","p","sclim"]].assign(split="test").to_csv(P.OUT/f"calib_{target}_test.csv",index=False)
R=pd.DataFrame(out_rows); S=pd.DataFrame(skill)
R.to_csv(P.OUT/"calib_yearly.csv",index=False); S.to_csv(P.OUT/"calib_skill.csv",index=False)
pd.set_option("display.width",220)
print(R.round(2).to_string(index=False)); print(S.round(3).T)


**`rev_figs.py`**: Figures 1-6.

In [ ]:
%%writefile /content/rev_pkg/rev_figs.py
"""Revision Figures 1-6 (non-trajectory). Inputs: CLASS_daily_events.csv and the
calibration output of rev_calib (calib_yearly.csv). Nothing upstream is recomputed."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_curve, roc_auc_score

import paths as P
from figstyle import (C_SUS, C_LOC, C_BOTH, C_NEI, C_ALL, SEG, SEG_COL,
                      LAB_SUS, LAB_LOC, shade_min)
import rev_maps as M
from rev_stats import load_class, load_hom

OUT = P.FIGS
REGIONS = M.REGIONS
RECEPTOR, ERA5_PT = M.RECEPTOR, M.ERA5_PT
REG_COL = dict(zip(REGIONS, ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd",
                             "#8c564b", "#7f7f7f", "#bcbd22", "#17becf"]))

def fig1():
    fig, ax = plt.subplots(figsize=(7.4, 5.6))
    M.basemap(ax, P.NE_DIR, lon=(20, 66), lat=(12, 40))
    ax.grid(True, alpha=0.25, ls=":")
    for name, (la0, la1, lo0, lo1) in REGIONS.items():
        c = REG_COL[name]
        ax.add_patch(Rectangle((lo0, la0), lo1 - lo0, la1 - la0, facecolor=c,
                               alpha=0.16, edgecolor=c, lw=1.3, zorder=1))
    ax.add_patch(Rectangle((35, 18), 20, 14, fill=False, edgecolor="k",
                           lw=1.4, ls="--", zorder=6))
    M.excl_circle(ax, color="tab:blue", lw=1.2)
    ax.plot(RECEPTOR[1], RECEPTOR[0], marker="*", ms=15, color="tab:blue",
            mec="k", mew=0.7, zorder=7, ls="none")
    ax.plot(ERA5_PT[1], ERA5_PT[0], marker="x", ms=6, color="k", mew=1.2,
            zorder=8, ls="none")
    handles = [Patch(facecolor=REG_COL[n], edgecolor=REG_COL[n], alpha=0.5,
                     label=n) for n in REGIONS]
    handles += [
        Line2D([], [], marker="*", ms=11, color="tab:blue", mec="k", ls="none",
               label="Receptor, Riyadh OERK"),
        Line2D([], [], color="tab:blue", ls=":", lw=1.2,
               label="300 km endpoint exclusion radius"),
        Line2D([], [], marker="x", color="k", ls="none",
               label="ERA5 receptor grid point (24.75°N, 46.75°E)"),
        Line2D([], [], color="k", ls="--", lw=1.4,
               label="ERA5 hourly domain (18–32°N, 35–55°E)"),
    ]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.02, 1.0),
              fontsize=7.5, borderaxespad=0)
    fig.savefig(OUT / "Fig1_study_region.png")
    plt.close(fig)


def fig2(d):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.9),
                             gridspec_kw={"width_ratios": [1.05, 1.15, 0.9]})
    cats = [("Sonly", C_SUS, "Suspension-only"),
            ("both", C_BOTH, "Overlap (both classes)"),
            ("Lonly", C_LOC, "Local-only"),
            ("neither", C_NEI, "Neither class")]
    m = d.groupby("month")[[c for c, _, _ in cats]].sum().reindex(range(1, 13), fill_value=0)
    bottom = np.zeros(12)
    for c, col, lab in cats:
        axes[0].bar(m.index, m[c], bottom=bottom, color=col, width=0.8, label=lab)
        bottom += m[c].values
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(list("JFMAMJJASOND"))
    axes[0].set_xlabel("Month")
    axes[0].set_ylabel("Dust days, 2016–2025")
    axes[0].set_title("(a) Seasonal cycle")

    y = d.groupby("year")[[c for c, _, _ in cats]].sum()
    bottom = np.zeros(len(y))
    for c, col, lab in cats:
        axes[1].bar(y.index, y[c], bottom=bottom, color=col, width=0.75)
        bottom += y[c].values
    axes[1].set_xticks(y.index)
    axes[1].set_xticklabels(y.index, rotation=45)
    axes[1].set_ylabel("Dust days per year")
    axes[1].set_title("(b) Annual counts")
    shade_min(axes[1])

    a = d.groupby("year")[["S", "L"]].sum()
    share = a.S / (a.S + a.L)
    axes[2].plot(a.index, share, "o-", color="#6a3d9a")
    axes[2].axhline(0.5, color="0.6", ls=":", lw=1)
    axes[2].set_ylim(0, 1)
    axes[2].set_xticks(a.index[::2])
    axes[2].set_xlabel("Year")
    axes[2].set_ylabel("S / (S + L)")
    axes[2].set_title("(c) “Suspension-dominated” share of class days")
    shade_min(axes[2])
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout()
    fig.savefig(OUT / "Fig2_event_climatology.png")
    plt.close(fig)


def fig3():
    h = load_hom()
    fig, axes = plt.subplots(3, 1, figsize=(6.8, 7.6), sharex=True)
    axes[0].plot(h.year, h.dust_rate, "o-", color="#c0392b", label="Dust codes")
    axes[0].plot(h.year, h.nondust_wx_rate, "s--", color="#2c7fb8",
                 label="Non-dust weather codes (rain, TS, haze)")
    axes[0].set_ylabel("Fraction of\nhourly reports")
    axes[0].set_title("(a) Dust and non-dust weather coding")
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[1].plot(h.year, h.vis_low_hours, "o-", color="#2ca25f", label="V < 5 km")
    axes[1].plot(h.year, h.vis_storm_hours, "s--", color="#006d2c", label="V < 1 km")
    axes[1].set_ylabel("Hours per year")
    axes[1].set_title("(b) Low-visibility hours (instrument record)")
    axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[2].plot(h.year, h.frac_lowvis_coded_dust, "o-", color="#e67e22")
    axes[2].set_ylim(0, 1)
    axes[2].set_ylabel("Fraction")
    axes[2].set_xlabel("Year")
    axes[2].set_title("(c) Share of V < 5 km hours carrying a dust code")
    for ax in axes:
        shade_min(ax)
    axes[2].set_xticks(h.year)
    fig.tight_layout()
    fig.savefig(OUT / "Fig3_record_homogeneity.png")
    plt.close(fig)


def fig4(d):
    d = d.copy()
    d["is_shamal"] = (d.wind_dir_vec >= 290) & (d.wind_dir_vec <= 350)
    d["shamal_wind"] = np.where(d.is_shamal, d.wind_speed_max, np.nan)
    a = d.groupby("year").agg(swvl1=("swvl1_mean", "mean"), p30=("precip_30d", "mean"),
                              shw=("shamal_wind", "mean"), shf=("is_shamal", "mean"))
    h = load_hom().set_index("year")
    fig, axes = plt.subplots(4, 1, figsize=(6.8, 9.2), sharex=True)

    def seg_of(y):
        for n, lo, hi in SEG:
            if lo <= y <= hi:
                return n
    cols = [SEG_COL[seg_of(y)] for y in h.index]
    axes[0].bar(h.index, h.dust_rate, color=cols, width=0.75)
    axes[0].set_ylabel("Dust report\nfraction")
    axes[0].set_title("(a) Observed dust activity")
    handles = [Patch(color=SEG_COL[n]) for n, _, _ in SEG]
    labels = ["Baseline 2016–2022", "Minimum 2023–2024", "Recovery 2025"]
    axes[0].legend(handles, labels, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    axes[0].set_ylim(0, 0.10)

    axes[1].plot(h.index, h.vis_low_hours, "o-", color="#2ca25f", label="V < 5 km")
    axes[1].plot(h.index, h.vis_storm_hours, "s--", color="#006d2c", label="V < 1 km")
    axes[1].set_ylabel("Hours per year")
    axes[1].set_title("(b) Low-visibility hours (independent of coding)")
    axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))

    l1, = axes[2].plot(a.index, a.swvl1, "o-", color="#2c7fb8", label="Soil moisture (0–7 cm)")
    ax2b = axes[2].twinx()
    l2, = ax2b.plot(a.index, a.p30, "^--", color="#41b6c4", label="30-day precipitation")
    ax2b.grid(False)
    axes[2].set_ylabel("Soil moisture\n(m$^3$ m$^{-3}$)")
    ax2b.set_ylabel("30-day precip. (mm)")
    axes[2].set_title("(c) Moisture pathway (ERA5, receptor grid point)")
    axes[2].legend([l1, l2], [l1.get_label(), l2.get_label()], loc="upper left",
                   bbox_to_anchor=(1.20, 1.0))

    l3, = axes[3].plot(a.index, a.shw, "o-", color="#238b45", label="Shamal wind speed")
    ax3b = axes[3].twinx()
    l4, = ax3b.plot(a.index, a.shf, "^:", color="#00441b", label="Shamal day fraction")
    ax3b.grid(False)
    axes[3].set_ylabel("Shamal wind\n(m s$^{-1}$)")
    ax3b.set_ylabel("Shamal day fraction")
    axes[3].set_xlabel("Year")
    axes[3].set_title("(d) Wind pathway (ERA5, receptor grid point)")
    axes[3].legend([l3, l4], [l3.get_label(), l4.get_label()], loc="upper left",
                   bbox_to_anchor=(1.20, 1.0))
    for ax in axes:
        shade_min(ax)
    axes[3].set_xticks(h.index)
    axes[3].set_xticklabels(h.index, rotation=45)
    fig.tight_layout()
    fig.savefig(OUT / "Fig4_regime_structure.png")
    plt.close(fig)
    return a


BASE = ["wind_speed_max", "swvl1_mean", "precip_sum", "precip_7d", "precip_30d",
        "vpd_max", "blh_max", "t2m_max", "ssrd_mean"]
F = BASE + ["wind_dir_sin", "wind_dir_cos", "wind_projection", "doy_sin", "doy_cos"]
PRETTY = {"wind_speed_max": "Daily max. wind speed", "wind_projection": "Wind projection (320°)",
          "vpd_max": "Daily max. VPD", "doy_sin": "Day-of-year (sin)", "doy_cos": "Day-of-year (cos)",
          "swvl1_mean": "Soil moisture", "t2m_max": "Daily max. temperature",
          "precip_30d": "30-day precipitation", "precip_7d": "7-day precipitation",
          "precip_sum": "Daily precipitation", "ssrd_mean": "Solar radiation",
          "blh_max": "Max. boundary-layer height", "wind_dir_sin": "Wind direction (sin)",
          "wind_dir_cos": "Wind direction (cos)"}


def features(d):
    d = d.copy()
    d["doy"] = d.date.dt.dayofyear
    th = np.radians(d.wind_dir_vec)
    d["wind_dir_sin"], d["wind_dir_cos"] = np.sin(th), np.cos(th)
    d["wind_projection"] = d.wind_speed_max * np.cos(th - np.radians(320.0))
    d["doy_sin"] = np.sin(2 * np.pi * d.doy / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * d.doy / 365.25)
    return d


def rf():
    return RandomForestClassifier(n_estimators=400, min_samples_leaf=5, class_weight="balanced",
                                  random_state=0, n_jobs=-1)


def fig5(d):
    d = features(d)
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.0),
                             gridspec_kw={"width_ratios": [1, 1.15, 0.95]})
    bins = np.arange(1.5, 13.5, 1.5)
    d["wbin"] = pd.cut(d.wind_speed_max, bins)
    g = d.groupby("wbin", observed=True).agg(L=("local_day", "mean"), S=("transport_day", "mean"),
                                             n=("local_day", "size")).reset_index()
    g = g[g.n >= 20]
    c = [iv.mid for iv in g.wbin]
    axes[0].plot(c, g.S, "s-", color=C_SUS, label="“Suspension-dominated”")
    axes[0].plot(c, g.L, "o-", color=C_LOC, label="Locally raised")
    axes[0].set_xlabel("Daily maximum 10 m wind speed (m s$^{-1}$)")
    axes[0].set_ylabel("Fraction of days in class")
    axes[0].set_title("(a) Response to local wind")
    axes[0].legend(loc="upper left")

    tr, te = d[d.year <= 2022], d[d.year >= 2023]
    m = rf().fit(tr[F].values, tr.local_day.values)
    perm = permutation_importance(m, te[F].values, te.local_day.values, n_repeats=20,
                                  random_state=0, scoring="roc_auc", n_jobs=-1)
    imp = pd.DataFrame({"f": F, "imp": perm.importances_mean, "sd": perm.importances_std})
    imp["rho"] = [stats.spearmanr(d[f], d.local_day)[0] for f in F]
    imp = imp.nlargest(8, "imp").sort_values("imp")
    cols = ["#2c7fb8" if r > 0 else "#c0392b" for r in imp.rho]
    axes[1].barh([PRETTY[f] for f in imp.f], imp.imp, xerr=imp.sd, color=cols,
                 error_kw=dict(lw=0.8))
    axes[1].set_xlabel("Permutation importance (drop in held-out AUC)")
    axes[1].set_title("(b) Drivers of locally raised days")
    axes[1].legend(handles=[Line2D([], [], color="#2c7fb8", lw=6, label="positive marginal ρ"),
                            Line2D([], [], color="#c0392b", lw=6, label="negative marginal ρ")],
                   loc="lower right")
    imp.to_csv(P.OUT / "fig5_importance_local.csv", index=False)

    for tgt, lab, col in [("local_day", "Locally raised", C_LOC),
                          ("transport_day", "“Susp.-dominated”", C_SUS)]:
        mm = rf().fit(tr[F].values, tr[tgt].values)
        s = mm.predict_proba(te[F].values)[:, 1]
        fpr, tpr, _ = roc_curve(te[tgt], s)
        axes[2].plot(fpr, tpr, color=col, label=f"{lab}, RF (AUC {roc_auc_score(te[tgt], s):.3f})")
        clim = te.month.map(tr.groupby("month")[tgt].mean())
        fpr, tpr, _ = roc_curve(te[tgt], clim)
        axes[2].plot(fpr, tpr, color=col, ls="--", lw=1,
                     label=f"{lab}, seasonal clim. (AUC {roc_auc_score(te[tgt], clim):.3f})")
    axes[2].plot([0, 1], [0, 1], ls=":", color="0.6", lw=1)
    axes[2].set_xlabel("False positive rate")
    axes[2].set_ylabel("True positive rate")
    axes[2].set_title("(c) Held-out skill, 2023–2025")
    axes[2].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=7, ncol=1)
    axes[2].set_aspect("equal")
    fig.tight_layout()
    fig.savefig(OUT / "Fig5_class_skill.png")
    plt.close(fig)
    return imp


def fig6():
    r = pd.read_csv(P.OUT / "calib_yearly.csv")
    fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.9), sharey=False)
    for ax, (cls, col, title) in zip(axes, [("Locally lifted", C_LOC, "(a) Locally raised"),
                                            ("Suspension-dominated", C_SUS, "(b) “Suspension-dominated”"),
                                            ("All dust", C_ALL, "(c) All dust days")]):
        s = r[r.cls == cls].sort_values("year")
        ax.fill_between(s.year, s.PI_lo, s.PI_hi, color=col, alpha=0.18, lw=0,
                        label="Calibrated model, 95% PI")
        ax.plot(s.year, s.E_model, "--", color=col, lw=1.6, label="Calibrated model, expected")
        ax.plot(s.year, s.E_sclim, ":", color="0.35", lw=1.2, label="Seasonal climatology")
        ax.plot(s.year, s.O, "o-", color="k", lw=1.6, ms=4.5, label="Observed")
        ax.axvline(2022.5, color="0.5", lw=0.8)
        shade_min(ax)
        ax.set_xticks(range(2016, 2026))
        ax.set_xticklabels(range(2016, 2026), rotation=45)
        ax.set_ylim(0, max(s.PI_hi.max(), s.O.max()) * 1.08)
        ax.set_ylabel("Days per year")
        ax.set_title(title)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 1.0))
    fig.tight_layout()
    fig.savefig(OUT / "Fig6_calibrated_expectation.png")
    plt.close(fig)




def make_all():
    d = load_class()
    fig1()
    fig2(d)
    fig3()
    fig4(d)
    fig5(d)
    fig6()
    print("Figures 1-6 written to", OUT)


**`rev_traj.py`**: Trajectory statistics: loading, class labels, Polissar weight, regions.

In [ ]:
%%writefile /content/rev_pkg/rev_traj.py
"""
Trajectory-statistics revision analyses (Reviewer 3, major comments 1, 2 and 4).

Works entirely from the sufficient statistics exported by cell R0
(REV_traj_cellcounts.npz: per-trajectory endpoint counts on the 1 deg grid,
after the 3 h and 300 km filters), the daily class table
(CLASS_daily_events.csv) and, optionally, the compact METAR file
(REV_metar_compact.csv.gz). HYSPLIT is not rerun.

Definitions (numbered as in the revised Section 2.7)
  n_ij      retained endpoints of all trajectories in the design sample   (Eq. 1)
  m_ij      retained endpoints of trajectories arriving on class days     (Eq. 1)
  p         class base rate = class trajectories / design trajectories    (Section 2.7)
  E_ij      = m_ij / (p n_ij) * W(n_ij)                                   (Section 2.7)
  E_ij^S    = sum_b m_ij,b / sum_b p_b n_ij,b * W(n_ij)   (stratum-standardized, Eq. 4)
  CWT_ij^c  = sum_k I_k^c DeltaC_k tau_ijk / sum_k tau_ijk                (Eq. 2)
  W(n)      Polissar weight keyed to the median occupied-cell count       (Eq. 3)
Expected value of E and E^S under random assignment of class days within
strata is 1 by construction; the event-count-pooled wind-matched field of the
submitted version is also reproduced (Epool) for comparison.
"""
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

RECEPTOR = (24.9578, 46.6989)
EXCL_KM = 300.0
SEASON = [3, 4, 5, 6]
PRIMARY = (2016, 2022)
REGIONS = {
    "Syrian Desert": (31.0, 35.5, 36.0, 41.0),
    "An Nafud": (27.0, 30.5, 38.0, 43.0),
    "Rub al Khali": (16.0, 23.0, 45.0, 56.0),
    "Mesopotamia / Fertile Crescent": (30.0, 37.0, 38.0, 49.0),
    "Ad-Dahna corridor": (21.0, 28.0, 44.0, 48.5),
    "Northeast Africa": (15.0, 32.0, 25.0, 37.0),
    "Eastern Arabia / Gulf": (24.0, 30.0, 47.0, 53.0),
    "Iranian plateau / Sistan": (28.0, 35.0, 54.0, 63.0),
}
THRESH = 1.5
EXT_FLOOR = 3.912 / 9.994  # clear-sky floor of the censored extinction (km-1)


# ------------------------------------------------------------------ loading
def load_counts(npz_path):
    z = np.load(npz_path)
    lat_e, lon_e = z["lat_edges"], z["lon_edges"]
    ncell = (len(lat_e) - 1) * (len(lon_e) - 1)
    tags = z["tags"].astype(str)
    C = sparse.csr_matrix((z["n"].astype(np.float64), (z["ti"], z["cell"].astype(np.int64))),
                          shape=(len(tags), ncell))
    meta = pd.DataFrame({"tag": tags, "n_all": z["n_all"], "n_age_ok": z["n_age_ok"],
                         "n_age_dist_ok": z["n_age_dist_ok"], "n_kept": z["n_kept"]})
    meta["date"] = pd.to_datetime(meta.tag.str[:8], format="%Y%m%d")
    meta["utc_hour"] = meta.tag.str[9:11].astype(int)
    meta["year"] = meta.date.dt.year
    meta["month"] = meta.date.dt.month
    return C, meta, lat_e, lon_e


def attach_days(meta, class_csv):
    d = pd.read_csv(class_csv, parse_dates=["date"])
    keep = ["date", "local_day", "transport_day", "dust_day", "wind_speed_max",
            "extinction_max", "vis_min_km", "local_hours", "transport_hours", "dust_hours"]
    d = d[[c for c in keep if c in d.columns]]
    m = meta.merge(d, on="date", how="left")
    m["L"] = (m.local_day == 1).astype(float)
    m["S"] = (m.transport_day == 1).astype(float)
    m["Lonly"] = ((m.L == 1) & (m.S == 0)).astype(float)
    m["Sonly"] = ((m.S == 1) & (m.L == 0)).astype(float)
    m["both"] = ((m.S == 1) & (m.L == 1)).astype(float)
    m["D"] = (m.dust_day == 1).astype(float)
    m.loc[m.local_day.isna(), ["L", "S", "Lonly", "Sonly", "both", "D"]] = np.nan
    return m


def dust_hour_intensity(meta, metar_csv, window_h=3):
    """Arrival-matched dust-hour extinction for each trajectory.

    For a trajectory arriving at t (00/06/12/18 UTC = 03/09/15/21 local), take
    the reports within +/- window_h hours that carry a dust code and return the
    maximum Koschmieder extinction among them; trajectories with no dust-coded
    report in the window get the clear-sky floor. Also returns the maximum
    over all dust-coded hours of the local day.
    """
    h = pd.read_csv(metar_csv, low_memory=False)
    h["valid"] = pd.to_datetime(h["valid"], utc=True, errors="coerce")
    h = h.dropna(subset=["valid"])
    toks = h["wxcodes"].fillna("").astype(str).str.split()
    dust = {"DU", "BLDU", "DS", "SA", "BLSA", "SS", "PO", "VCBLDU", "VCBLSA", "VCDS", "VCSS",
            "+DS", "+SS", "-DS", "-SS"}
    h["is_dust"] = toks.apply(lambda t: bool(dust.intersection(t)))
    h["ext"] = 3.912 / h["vis_km"].clip(lower=0.05)
    hd = h[h.is_dust].copy()
    hd["local_date"] = (hd.valid + pd.Timedelta(hours=3)).dt.tz_localize(None).dt.normalize()
    day_max = hd.groupby("local_date")["ext"].max()
    arr = (meta.date + pd.to_timedelta(meta.utc_hour, unit="h")).dt.tz_localize("UTC")
    t = hd.valid.values.astype("datetime64[s]").astype(np.int64)
    e = hd.ext.values
    order = np.argsort(t)
    t, e = t[order], e[order]
    a = arr.values.astype("datetime64[s]").astype(np.int64)
    lo = np.searchsorted(t, a - window_h * 3600, side="left")
    hi = np.searchsorted(t, a + window_h * 3600, side="right")
    out = np.full(len(a), EXT_FLOOR)
    for i in np.where(hi > lo)[0]:
        out[i] = e[lo[i]:hi[i]].max()
    return out, meta.date.map(day_max).fillna(EXT_FLOOR).values


# ----------------------------------------------------------------- weights
def polissar(n):
    occ = n[n > 0]
    base = np.median(occ) if occ.size else np.nan
    w = np.full(n.shape, 0.05)
    w[n > base] = 0.42
    w[n > 1.5 * base] = 0.70
    w[n > 3.0 * base] = 1.00
    w[n == 0] = 0.0
    return w, base


# ------------------------------------------------------------------ fields
def fields_for(C, lab, strata=None, weights=None):
    """Return dict of PSCF/enrichment fields for one class label vector.

    lab     : 0/1 class indicator per trajectory in the design sample
    strata  : integer stratum per trajectory (None = single stratum)
    weights : per-trajectory intensity (DeltaC_k) for the CWT-type field
    """
    lab = np.asarray(lab, float)
    ntraj = np.ones_like(lab)
    n = np.asarray(C.sum(axis=0)).ravel()
    m = np.asarray(C.T @ lab).ravel()
    w, base = polissar(n)
    out = {"n": n, "m": m, "w": w, "base_n": base}
    p = lab.sum() / ntraj.sum()
    with np.errstate(divide="ignore", invalid="ignore"):
        out["pscf"] = np.where(n > 0, m / n, 0.0)
        out["E"] = np.where(n > 0, m / (p * n), 0.0) * w
        out["E_raw"] = np.where(n > 0, m / (p * n), np.nan)
    out["p"] = p
    if strata is not None:
        s = np.asarray(strata)
        exp = np.zeros_like(n)
        num_pool = np.zeros_like(n)
        den_pool = np.zeros_like(n)
        rows = []
        for b in np.unique(s[~pd.isna(s)]):
            sel = s == b
            Cb = C[np.where(sel)[0]]
            nb = np.asarray(Cb.sum(axis=0)).ravel()
            mb = np.asarray(Cb.T @ lab[sel]).ravel()
            pb = lab[sel].mean()
            exp += pb * nb
            nev, nct = int(lab[sel].sum()), int((1 - lab[sel]).sum())
            rows.append({"stratum": b, "n_event": nev, "n_control": nct, "p_b": pb})
            if nev >= 15 and nct >= 15:           # submitted-version pooling
                num_pool += mb * nev
                den_pool += nb * nev
        with np.errstate(divide="ignore", invalid="ignore"):
            out["ES"] = np.where(exp > 0, m / exp, 0.0) * w
            out["ES_raw"] = np.where(exp > 0, m / exp, np.nan)
            used = [r for r in rows if r["n_event"] >= 15 and r["n_control"] >= 15]
            pbar = (sum(r["n_event"] for r in used) /
                    max(1, sum(r["n_event"] + r["n_control"] for r in used)))
            out["Epool"] = np.where(den_pool > 0, num_pool / den_pool / pbar, 0.0) * w
        out["strata"] = pd.DataFrame(rows)
        out["expected"] = exp
    if weights is not None:
        cw = np.asarray(weights, float) * lab
        num = np.asarray(C.T @ cw).ravel()
        mu = cw.sum() / ntraj.sum()
        with np.errstate(divide="ignore", invalid="ignore"):
            out["cwt"] = np.where(n > 0, num / n, 0.0) * w                     # Eq. 2
            out["cwtE"] = np.where(n > 0, num / (mu * n), 0.0) * w             # normalised
        if strata is not None:
            s = np.asarray(strata)
            expc = np.zeros_like(n)
            for b in np.unique(s[~pd.isna(s)]):
                sel = s == b
                nb = np.asarray(C[np.where(sel)[0]].sum(axis=0)).ravel()
                expc += cw[sel].mean() * nb
            with np.errstate(divide="ignore", invalid="ignore"):
                out["cwtES"] = np.where(expc > 0, num / expc, 0.0) * w
    return out


def corr(a, b, mask):
    x, y = a[mask], b[mask]
    ok = np.isfinite(x) & np.isfinite(y)
    return float(np.corrcoef(x[ok], y[ok])[0, 1]) if ok.sum() > 20 else np.nan


# ------------------------------------------------------------ permutation
def permute_days(day_ids, day_lab, day_strata, rng):
    """Shuffle day-level labels within strata; returns new day labels."""
    new = day_lab.copy()
    for b in np.unique(day_strata):
        idx = np.where(day_strata == b)[0]
        new[idx] = rng.permutation(day_lab[idx])
    return new


def region_masks(lat_e, lon_e):
    lat_m = (lat_e[:-1] + lat_e[1:]) / 2
    lon_m = (lon_e[:-1] + lon_e[1:]) / 2
    LA, LO = np.meshgrid(lat_m, lon_m, indexing="ij")
    masks = {}
    for name, (a0, a1, o0, o1) in REGIONS.items():
        masks[name] = ((LA >= a0) & (LA <= a1) & (LO >= o0) & (LO <= o1)).ravel()
    masks["All eight regions"] = np.any(np.vstack(list(masks.values())), axis=0)
    masks["Whole domain"] = np.ones(LA.size, dtype=bool)
    dlat = np.radians(LA - RECEPTOR[0])
    dlon = np.radians(LO - RECEPTOR[1])
    a = (np.sin(dlat / 2) ** 2 + np.cos(np.radians(RECEPTOR[0])) * np.cos(np.radians(LA))
         * np.sin(dlon / 2) ** 2)
    dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    return masks, dist.ravel()


**`rev_traj_run.py`**: Trajectory re-analysis: designs, standardized enrichment, class-conditional CWT, permutation nulls.

In [ ]:
%%writefile /content/rev_pkg/rev_traj_run.py
"""Trajectory-statistics revision analyses (Reviewer 3, major comments 1, 2 and 4).

run_all(npz, class_csv, outdir, metar=None, B=1000) reproduces the submitted
fields from the exported sufficient statistics and adds:
  * mutually exclusive classes (local-only, suspension-only);
  * counts by month, season and wind quintile, in days and trajectories;
  * stratum-standardised enrichment E^S (null expectation 1) next to the
    event-count-pooled field of the submitted version;
  * class-conditional CWT with two (optionally four) intensity measures;
  * day-block permutation nulls within the strata of each design, for the
    between-class correlations (weighted and unweighted), every cell and
    every region, for PSCF and CWT.
HYSPLIT is not rerun; everything comes from REV_traj_cellcounts.npz.
"""
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

import rev_traj as R

warnings.filterwarnings("ignore", message="All-NaN slice encountered")
warnings.filterwarnings("ignore", message="Mean of empty slice")
warnings.filterwarnings("ignore", message="invalid value encountered")
CLASSES = ["L", "S", "Lonly", "Sonly"]
PAIRS = [("L", "S"), ("Lonly", "Sonly")]
DESIGNS = ["unmatched", "season", "wind"]


def _intensities(meta, metar):
    meta["dC_daymax"] = (meta.extinction_max - R.EXT_FLOOR).clip(lower=0).fillna(0)
    meta["dC_hours_L"] = meta.local_hours.fillna(0)
    meta["dC_hours_S"] = meta.transport_hours.fillna(0)
    kinds = ["daymax", "hours"]
    if metar is not None and Path(metar).exists():
        arr, dmax = R.dust_hour_intensity(meta, metar, window_h=3)
        meta["dC_arrival"] = np.clip(arr - R.EXT_FLOOR, 0, None)
        meta["dC_dustday"] = np.clip(dmax - R.EXT_FLOOR, 0, None)
        kinds += ["arrival", "dustday"]
    return kinds


def _intensity_vec(meta, c, k):
    if k == "hours":
        return meta["dC_hours_L" if c.startswith("L") else "dC_hours_S"].values
    return meta[f"dC_{k}"].values


def _std_field(C, v, strata_idx, nb_mat):
    """Observed-over-expected field for one per-trajectory vector v."""
    num = np.asarray(C.T @ v).ravel()
    means = np.array([v[strata_idx == j].mean() for j in range(nb_mat.shape[0])])
    exp = means @ nb_mat
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(exp > 0, num / exp, np.nan)


def run_all(npz, class_csv, outdir, metar=None, B=1000, seed=20260928, verbose=True):
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)

    C, meta, lat_e, lon_e = R.load_counts(npz)
    meta = R.attach_days(meta, class_csv)
    kinds = _intensities(meta, metar)
    masks, dist = R.region_masks(lat_e, lon_e)
    res = {"n_traj_files": int(len(meta)), "endpoints_all": int(meta.n_all.sum()),
           "endpoints_after_filters": int(meta.n_age_dist_ok.sum()),
           "endpoints_in_grid": int(meta.n_kept.sum()),
           "traj_without_endpoints_all_years": int((meta.n_age_dist_ok == 0).sum()),
           "intensities": kinds}

    # ------------------------------------------------------------ designs
    prim = meta.year.between(*R.PRIMARY) & meta.L.notna()
    keep = prim & (meta.n_age_dist_ok > 0)
    des = {"unmatched": keep.values, "season": (keep & meta.month.isin(R.SEASON)).values}
    des["wind"] = des["season"].copy()
    sel = np.where(des["wind"])[0]
    q = pd.qcut(meta.wind_speed_max.values[sel], 5, labels=False, duplicates="drop")
    strata = {d: np.zeros(len(meta), dtype=int) for d in DESIGNS}
    strata["wind"][:] = -1
    strata["wind"][sel] = q
    res["dropped_no_endpoint"] = {
        "all_months_2016_2022": int((prim & (meta.n_age_dist_ok == 0)).sum()),
        "MarJun_2016_2022": int((prim & meta.month.isin(R.SEASON) & (meta.n_age_dist_ok == 0)).sum())}
    res["design_sizes"] = {}
    for d in DESIGNS:
        s = meta[des[d]]
        res["design_sizes"][d] = {"traj": int(len(s)), "days": int(s.date.nunique()),
                                  **{f"{c}_traj": int(s[c].sum()) for c in CLASSES},
                                  **{f"{c}_days": int(s.drop_duplicates("date")[c].sum()) for c in CLASSES}}

    # ------------------------------------------------------------ counts (Table 5, S4)
    per = meta.iloc[sel].copy()
    per["q"] = q
    rows = []
    for b in range(int(q.max()) + 1):
        s = per[per.q == b]
        dd = s.drop_duplicates("date")
        r = {"quintile": b + 1, "wind_lo": float(s.wind_speed_max.min()), "wind_hi": float(s.wind_speed_max.max()),
             "days": int(len(dd)), "traj": int(len(s))}
        for c in ["S", "L", "both", "Sonly", "Lonly"]:
            r[f"{c}_days"] = int(dd[c].sum())
            r[f"{c}_traj"] = int(s[c].sum())
        r["p_S"] = r["S_traj"] / r["traj"]
        r["p_L"] = r["L_traj"] / r["traj"]
        r["nonS_traj"] = r["traj"] - r["S_traj"]
        r["nonL_traj"] = r["traj"] - r["L_traj"]
        rows.append(r)
    t5 = pd.DataFrame(rows)
    t5.to_csv(out / "table5_counts_by_quintile.csv", index=False)
    mon = per.drop_duplicates("date").groupby("month")[["S", "L", "both", "Sonly", "Lonly"]].sum().astype(int)
    mon.to_csv(out / "counts_by_month_MarJun.csv")
    allm = meta[des["unmatched"]].drop_duplicates("date").copy()
    allm["season_name"] = allm.month.map({12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
                                          6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"})
    allm.groupby("season_name")[["S", "L", "both", "Sonly", "Lonly"]].sum().astype(int).to_csv(
        out / "counts_by_season_2016_2022.csv")

    # ------------------------------------------------------------ observed fields
    F, design_cache = {}, {}
    for d in DESIGNS:
        idx = np.where(des[d])[0]
        Cd = C[idx].tocsr()
        st = strata[d][idx]
        sids = np.unique(st)
        st_idx = np.searchsorted(sids, st)
        nb_mat = np.vstack([np.asarray(Cd[st_idx == j].sum(axis=0)).ravel() for j in range(len(sids))])
        n = nb_mat.sum(axis=0)
        w, base_n = R.polissar(n)
        design_cache[d] = (idx, Cd, st_idx, nb_mat, n, w)
        for c in CLASSES:
            lab = meta[c].values[idx].astype(float)
            f = {"n": n, "w": w, "base_n": base_n, "p": float(lab.mean())}
            f["E_raw"] = _std_field(Cd, lab, st_idx, nb_mat)
            f["E"] = np.nan_to_num(f["E_raw"]) * w
            for k in kinds:
                v = lab * _intensity_vec(meta, c, k)[idx]
                f[f"cwt_{k}_raw"] = _std_field(Cd, v, st_idx, nb_mat)
                f[f"cwt_{k}"] = np.nan_to_num(f[f"cwt_{k}_raw"]) * w
                num = np.asarray(Cd.T @ v).ravel()
                with np.errstate(divide="ignore", invalid="ignore"):
                    f[f"cwt_{k}_abs"] = np.where(n > 0, num / n, 0.0) * w      # Eq. (2), in km-1 or hours
            if d == "wind":   # event-count-pooled field of the submitted version (Section 2.7)
                num_p = np.zeros_like(n); den_p = np.zeros_like(n); ne = nc = 0
                for j in range(len(sids)):
                    s = st_idx == j
                    nev, nct = int(lab[s].sum()), int((1 - lab[s]).sum())
                    if nev < 15 or nct < 15:
                        continue
                    mb = np.asarray(Cd[s].T @ lab[s]).ravel()
                    num_p += mb * nev; den_p += nb_mat[j] * nev; ne += nev; nc += nct
                pU = ne / max(1, ne + nc)
                # as in the submitted version: n, W and the occupancy mask use only the quintiles retained
                used_b = [j for j in range(len(sids)) if lab[st_idx == j].sum() >= 15
                          and (1 - lab[st_idx == j]).sum() >= 15]
                n_used = nb_mat[used_b].sum(axis=0)
                w_used, _ = R.polissar(n_used)
                with np.errstate(divide="ignore", invalid="ignore"):
                    f["Epool_raw"] = np.where(den_p > 0, num_p / den_p / pU, np.nan)
                f["Epool"] = np.nan_to_num(f["Epool_raw"]) * w_used
                f["n_used"] = n_used
                f["w_used"] = w_used
                # null expectation of the pooled field in each cell: sum_b N_cb p_b n_b / (sum_b N_cb n_b) / pU
                pb = np.array([lab[st_idx == j].mean() for j in range(len(sids))])
                Ncb = np.array([lab[st_idx == j].sum() for j in range(len(sids))])
                use = np.array([(Ncb[j] >= 15) and ((st_idx == j).sum() - Ncb[j] >= 15) for j in range(len(sids))])
                ex_num = ((Ncb * pb * use)[:, None] * nb_mat).sum(0)
                ex_den = ((Ncb * use)[:, None] * nb_mat).sum(0)
                with np.errstate(divide="ignore", invalid="ignore"):
                    f["Epool_nullexp"] = np.where(ex_den > 0, ex_num / ex_den / pU, np.nan)
            F[(d, c)] = f

    # ------------------------------------------------------------ observed correlations
    def r_w(a, b, fa, fb):
        return R.corr(a, b, (fa["n"] > 0) & (fb["n"] > 0))

    def r_u(a, b, f):
        return R.corr(a, b, (f["w"] >= 0.70) & (f["n"] > 0))

    corr_rows = []
    for a, b in [("L", "S"), ("Lonly", "Sonly"), ("L", "Lonly"), ("S", "Sonly")]:
        for d in DESIGNS:
            fa, fb = F[(d, a)], F[(d, b)]
            row = {"pair": f"{a}-{b}", "design": d,
                   "PSCF_weighted": r_w(fa["E"], fb["E"], fa, fb),
                   "PSCF_unweighted_W>=0.7": r_u(fa["E_raw"], fb["E_raw"], fa)}
            if d == "wind":
                row["PSCF_pooled_weighted"] = R.corr(fa["Epool"], fb["Epool"], (fa["n_used"] > 0) & (fb["n_used"] > 0))
            for k in kinds:
                row[f"CWT_{k}_weighted"] = r_w(fa[f"cwt_{k}"], fb[f"cwt_{k}"], fa, fb)
                row[f"CWT_{k}_unweighted_W>=0.7"] = r_u(fa[f"cwt_{k}_raw"], fb[f"cwt_{k}_raw"], fa)
            corr_rows.append(row)
    corr_df = pd.DataFrame(corr_rows)
    corr_df.to_csv(out / "class_field_correlations.csv", index=False)

    # ------------------------------------------------------------ permutation nulls
    fields_null = ["PSCF"] + [f"CWT_{k}" for k in kinds]
    null_r, null_rows, region_rows, sig, contrast_rows = {}, [], [], {}, []
    for d in DESIGNS:
        idx, Cd, st_idx, nb_mat, n, w = design_cache[d]
        sub = meta.iloc[idx]
        days, day_of = np.unique(sub.date.values, return_inverse=True)
        day_st = pd.Series(st_idx).groupby(day_of).first().values
        # day-level attributes (labels and intensities move together)
        vecs = {}
        for c in CLASSES:
            lab_d = pd.Series(sub[c].values).groupby(day_of).first().values.astype(float)
            vecs[("PSCF", c)] = lab_d
            for k in kinds:
                inten = pd.Series(_intensity_vec(meta, c, k)[idx]).groupby(day_of).first().values
                vecs[(f"CWT_{k}", c)] = lab_d * inten
        keys = list(vecs)
        Vd = np.vstack([vecs[kk] for kk in keys])                # nvec x ndays
        CdT = Cd.T.tocsr()
        ncell = Cd.shape[1]
        occ = n > 0
        acc = {kk: np.empty((B, ncell), dtype=np.float32) for kk in keys}
        groups = [np.where(day_st == g)[0] for g in np.unique(day_st)]
        chunk = 50
        for k0 in range(0, B, chunk):
            kk_n = min(chunk, B - k0)
            blocks = []
            for _ in range(kk_n):
                src = np.arange(len(days))
                for g in groups:
                    src[g] = g[rng.permutation(len(g))]
                blocks.append(Vd[:, src][:, day_of])             # nvec x ntraj
            Vt = np.concatenate(blocks, axis=0)                  # (kk_n*nvec) x ntraj
            M = (CdT @ Vt.T).T                                   # (kk_n*nvec) x ncell
            means = np.stack([Vt[:, st_idx == j].mean(axis=1) for j in range(nb_mat.shape[0])], axis=1)
            EXP = means @ nb_mat
            with np.errstate(divide="ignore", invalid="ignore"):
                Er = np.where(EXP > 0, M / EXP, np.nan).astype(np.float32)
            Er = Er.reshape(kk_n, len(keys), ncell)
            for i, kk in enumerate(keys):
                acc[kk][k0:k0 + kk_n] = Er[:, i, :]
        # correlations under the null
        for fld in fields_null:
            for a, b in PAIRS:
                Na, Nb = acc[(fld, a)], acc[(fld, b)]
                rw = np.array([R.corr(np.nan_to_num(Na[k]) * w, np.nan_to_num(Nb[k]) * w, occ) for k in range(B)])
                ru = np.array([R.corr(Na[k], Nb[k], (w >= 0.70) & occ) for k in range(B)])
                null_r[f"{d}__{fld}__{a}-{b}__weighted"] = rw
                null_r[f"{d}__{fld}__{a}-{b}__unweighted"] = ru
                ka = "E" if fld == "PSCF" else "cwt_" + fld.split("_", 1)[1]
                fa, fb = F[(d, a)], F[(d, b)]
                ow = r_w(fa[ka], fb[ka], fa, fb)
                ou = r_u(fa[ka + "_raw"], fb[ka + "_raw"], fa)
                null_rows.append({
                    "design": d, "field": fld, "pair": f"{a}-{b}",
                    "r_obs_weighted": ow, "null_w_mean": np.nanmean(rw),
                    "null_w_lo": np.nanpercentile(rw, 2.5), "null_w_hi": np.nanpercentile(rw, 97.5),
                    "p_w_greater": (np.sum(rw >= ow) + 1) / (B + 1),
                    "r_obs_unweighted": ou, "null_u_mean": np.nanmean(ru),
                    "null_u_lo": np.nanpercentile(ru, 2.5), "null_u_hi": np.nanpercentile(ru, 97.5),
                    "p_u_greater": (np.sum(ru >= ou) + 1) / (B + 1)})
        # cells and regions
        for fld in fields_null:
            for c in CLASSES:
                f = F[(d, c)]
                ka = "E" if fld == "PSCF" else "cwt_" + fld.split("_", 1)[1]
                Eo_raw, Eo = f[ka + "_raw"], f[ka]
                Nn = acc[(fld, c)]
                q95 = np.nanpercentile(Nn, 95, axis=0)
                s_cell = (Eo_raw > q95) & occ
                sig[(d, fld, c)] = s_cell
                Nw = np.nan_to_num(Nn) * w
                null_sig_frac = (Nn > q95[None, :])
                for reg, rm in masks.items():
                    o = rm & occ
                    if o.sum() == 0:
                        continue
                    obs_mean = float(np.mean(Eo[o]))
                    null_means = Nw[:, o].mean(axis=1)
                    cross = np.asarray(Cd[:, np.where(rm)[0]].sum(axis=1)).ravel() > 0
                    subc = sub[cross]
                    region_rows.append({
                        "design": d, "field": fld, "class": c, "region": reg,
                        "cells_occupied": int(o.sum()), "cells_W1": int((o & (w >= 1.0)).sum()),
                        "cells_W>=0.7": int((o & (w >= 0.70)).sum()),
                        "median_n_ij": float(np.median(n[o])),
                        "endpoint_share": float(n[rm].sum() / n.sum()),
                        "excluded_frac": float((dist[rm] < R.EXCL_KM).mean()),
                        "traj_crossing": int(cross.sum()),
                        "class_traj_crossing": int(subc[c].sum()),
                        "class_days_crossing": int(subc.drop_duplicates("date")[c].sum()),
                        "mean": obs_mean, "frac_gt_1p5": float((Eo[o] > R.THRESH).mean()),
                        "null_mean": float(np.mean(null_means)),
                        "null_lo": float(np.percentile(null_means, 2.5)),
                        "null_hi": float(np.percentile(null_means, 97.5)),
                        "p_region": float((np.sum(null_means >= obs_mean) + 1) / (B + 1)),
                        "null_frac_gt_1p5": float(np.mean((Nw[:, o] > R.THRESH).mean(axis=1))),
                        "frac_cells_sig": float(s_cell[o].mean()),
                        "null_frac_cells_sig": float(null_sig_frac[:, o].mean()),
                        "null_frac_cells_sig_hi": float(np.percentile(null_sig_frac[:, o].mean(axis=1), 97.5)),
                        "p_field_sig": float((np.sum(null_sig_frac[:, o].mean(axis=1) >= s_cell[o].mean()) + 1)
                                             / (B + 1)),
                        "mean_unweighted_W>=0.42": float(np.nanmean(Eo_raw[o & (w >= 0.42)]))
                        if (o & (w >= 0.42)).any() else np.nan,
                    })
        # negative-control contrasts: suspension minus local, region means, joint-permutation null
        for fld in fields_null:
            ka = "E" if fld == "PSCF" else "cwt_" + fld.split("_", 1)[1]
            for a, b in [("S", "L"), ("Sonly", "Lonly")]:
                Na = np.nan_to_num(acc[(fld, a)]) * w
                Nb = np.nan_to_num(acc[(fld, b)]) * w
                for reg, rm in masks.items():
                    o = rm & occ
                    if o.sum() == 0:
                        continue
                    obs = float(F[(d, a)][ka][o].mean() - F[(d, b)][ka][o].mean())
                    nd = Na[:, o].mean(axis=1) - Nb[:, o].mean(axis=1)
                    contrast_rows.append({
                        "design": d, "field": fld, "contrast": f"{a} minus {b}", "region": reg,
                        "obs_diff": obs, "null_mean": float(nd.mean()),
                        "null_lo": float(np.percentile(nd, 2.5)), "null_hi": float(np.percentile(nd, 97.5)),
                        "p_susp_greater": float((np.sum(nd >= obs) + 1) / (B + 1)),
                        "p_local_greater": float((np.sum(nd <= obs) + 1) / (B + 1))})
        if verbose:
            print(f"  permutation nulls done: {d}")
    null_df = pd.DataFrame(null_rows)
    null_df.to_csv(out / "permutation_null_correlations.csv", index=False)
    reg_df = pd.DataFrame(region_rows)
    reg_df.to_csv(out / "region_summary_with_nulls.csv", index=False)
    np.savez_compressed(out / "null_correlations.npz", **null_r)
    pd.DataFrame(contrast_rows).to_csv(out / "negative_control_contrasts.csv", index=False)

    # ------------------------------------------------------------ submitted-version reproduction
    rep = []
    for c in ["S", "L"]:
        f = F[("wind", c)]
        for reg, rm in masks.items():
            o = rm & (f["n"] > 0)
            ou = rm & (f["n_used"] > 0)
            if o.sum():
                rep.append({"class": c, "region": reg, "pooled_mean": float(f["Epool"][ou].mean()),
                            "pooled_frac_gt_1p5": float((f["Epool"][ou] > 1.5).mean()),
                            "pooled_null_expectation_mean": float(np.nanmean(f["Epool_nullexp"][ou])),
                            "pooled_cells": int(ou.sum()),
                            "pooled_endpoint_share": float(f["n_used"][rm].sum() / f["n_used"].sum()),
                            "std_mean": float(f["E"][o].mean()),
                            "std_frac_gt_1p5": float((f["E"][o] > 1.5).mean()),
                            "std_mean_null": float(reg_df[(reg_df.design == "wind") & (reg_df.field == "PSCF")
                                                          & (reg_df["class"] == c) & (reg_df.region == reg)]
                                                   ["null_mean"].iloc[0]),
                            "cells": int(o.sum()), "endpoint_share": float(f["n"][rm].sum() / f["n"].sum())})
    pd.DataFrame(rep).to_csv(out / "pooled_vs_standardised_regions.csv", index=False)
    res["pooled_null_expectation_domain_mean"] = {
        c: float(np.nanmean(F[("wind", c)]["Epool_nullexp"][F[("wind", c)]["n_used"] > 0])) for c in CLASSES}

    # ------------------------------------------------------------ save fields
    save = {"lat_edges": lat_e, "lon_edges": lon_e}
    for (d, c), f in F.items():
        for k, v in f.items():
            if isinstance(v, np.ndarray):
                save[f"{d}__{c}__{k}"] = v
    for (d, fld, c), v in sig.items():
        save[f"sig__{d}__{fld}__{c}"] = v
    np.savez_compressed(out / "fields_revision.npz", **save)
    res["base_rates"] = {f"{d}:{c}": F[(d, c)]["p"] for (d, c) in F}
    res["base_n_median"] = {d: float(F[(d, "L")]["base_n"]) for d in DESIGNS}
    res["wind_quintile_edges"] = t5[["wind_lo", "wind_hi"]].round(3).values.tolist()
    json.dump(res, open(out / "summary.json", "w"), indent=1, default=float)
    if verbose:
        print(json.dumps(res, indent=1, default=float)[:2500])
        print(corr_df.round(3).to_string())
        print(null_df.round(3).to_string())
    return res


**`rev_traj_figs.py`**: Figures 7-8.

In [ ]:
%%writefile /content/rev_pkg/rev_traj_figs.py
"""Figures 7-8 of the revision (trajectory statistics).

Inputs: the output directory of rev_traj_run.run_all (CSV/NPZ). Nothing is recomputed.
"""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import figstyle  # noqa: F401  (rcParams)
from figstyle import C_SUS, C_LOC, C_BOTH
import rev_maps as M

DES_LAB = {"unmatched": "Unmatched\n(all months)", "season": "Season-matched\n(Mar–Jun)",
           "wind": "Wind-matched\n(Mar–Jun, quintiles)"}
PAIR_STYLE = {"L-S": dict(color=C_BOTH, marker="o", label="Overlapping classes (locally raised vs “suspension-dominated”)"),
              "Lonly-Sonly": dict(color="#1b7837", marker="s", label="Mutually exclusive classes (local-only vs suspension-only)")}
CLS_LAB = {"S": "“Suspension-dominated”", "L": "Locally raised", "Sonly": "Suspension-only", "Lonly": "Local-only"}


def _g(z, key):
    return z[key] if key in z.files else None


def _load(res_dir):
    res_dir = Path(res_dir)
    z = np.load(res_dir / "fields_revision.npz")
    nul = pd.read_csv(res_dir / "permutation_null_correlations.csv")
    cor = pd.read_csv(res_dir / "class_field_correlations.csv")
    return z, nul, cor


def fig7(res_dir, out_png, fields=(("PSCF", "PSCF"), ("CWT_daymax", "CWT"))):
    """Between-class correlations and the negative-control contrast, each against its permutation null."""
    res_dir = Path(res_dir)
    nul = pd.read_csv(res_dir / "permutation_null_correlations.csv")
    con = pd.read_csv(res_dir / "negative_control_contrasts.csv")
    con = con[con.region == "All eight regions"]
    cols = [("w", "weighted r"), ("u", "unweighted r"), ("c", "contrast")]
    designs = ["unmatched", "season", "wind"]
    fig, axes = plt.subplots(len(fields), 3, figsize=(7.4, 2.55 * len(fields) + 0.75), sharex=True, squeeze=False)
    k = 0
    for i, (fld, flab) in enumerate(fields):
        for j, (kind, clab) in enumerate(cols):
            ax = axes[i, j]
            for jj, pair in enumerate(["L-S", "Lonly-Sonly"]):
                st = PAIR_STYLE[pair]
                dx = -0.15 if jj == 0 else 0.15
                for x, d in enumerate(designs):
                    if kind == "c":
                        cname = "S minus L" if pair == "L-S" else "Sonly minus Lonly"
                        r = con[(con.design == d) & (con.field == fld) & (con.contrast == cname)]
                        if r.empty:
                            continue
                        r = r.iloc[0]
                        lo, hi, mu, obs = r.null_lo, r.null_hi, r.null_mean, r.obs_diff
                    else:
                        r = nul[(nul.design == d) & (nul.field == fld) & (nul.pair == pair)]
                        if r.empty:
                            continue
                        r = r.iloc[0]
                        lo, hi, mu = r[f"null_{kind}_lo"], r[f"null_{kind}_hi"], r[f"null_{kind}_mean"]
                        obs = r["r_obs_weighted" if kind == "w" else "r_obs_unweighted"]
                    ax.add_patch(plt.Rectangle((x + dx - 0.09, lo), 0.18, hi - lo, color="0.82", lw=0, zorder=1))
                    ax.plot([x + dx - 0.09, x + dx + 0.09], [mu, mu], color="0.35", lw=1.0, zorder=2)
                    ax.plot(x + dx, obs, marker=st["marker"], color=st["color"], ms=5.5, mec="k", mew=0.5,
                            zorder=3, ls="none")
            ax.axhline(0, color="0.5", lw=0.6)
            ax.set_xlim(-0.5, 2.5)
            ax.set_ylim((-0.45, 0.45) if kind == "c" else (-0.4, 1.0))
            ax.set_xticks(range(3))
            ax.set_xticklabels(["Unmatched", "Season", "Wind"], fontsize=7.5)
            ax.set_title(f"({'abcdefghi'[k]}) {flab}, {clab}", loc="left", fontsize=8)
            k += 1
        axes[i, 0].set_ylabel("Correlation, r")
        axes[i, 2].set_ylabel("S − L, mean enrichment")
    for ax in axes[-1]:
        ax.set_xlabel("Background design", fontsize=8)
    handles = [Line2D([], [], marker=s["marker"], color=s["color"], mec="k", mew=0.5, ls="none", ms=6,
                      label=s["label"]) for s in PAIR_STYLE.values()]
    handles += [Patch(color="0.82", label="Permutation null, central 95% (line: mean)")]
    fig.legend(handles=handles, loc="lower center", ncol=1, bbox_to_anchor=(0.5, -0.005), fontsize=7.5)
    fig.tight_layout(rect=(0, 0.115 if len(fields) == 2 else 0.075, 1, 1), h_pad=1.0, w_pad=0.6)
    fig.savefig(out_png)
    plt.close(fig)


def _map_panel(ax, z, key_raw, key_w, sig, title, ne_dir, norm, cmap, show_sparse=True, labels=(True, True)):
    lat_e, lon_e = z["lat_edges"], z["lon_edges"]
    ny, nx = len(lat_e) - 1, len(lon_e) - 1
    E = z[key_raw].reshape(ny, nx)
    w = z[key_w].reshape(ny, nx)
    good = np.isfinite(E) & (w >= 0.42)
    sparse_ = (w > 0) & (w < 0.42)
    if show_sparse:
        ax.pcolormesh(lon_e, lat_e, np.where(sparse_, 1.0, np.nan), cmap=ListedColormap(["0.88"]), zorder=1)
    Ep = np.where(good, np.clip(E, norm.vmin * 1.0001, norm.vmax * 0.9999), np.nan)
    pm = ax.pcolormesh(lon_e, lat_e, Ep, cmap=cmap, norm=norm, zorder=2)
    if sig is not None:
        s = sig.reshape(ny, nx) & good
        yy, xx = np.where(s)
        ax.plot(lon_e[xx] + 0.5, lat_e[yy] + 0.5, ".", color="k", ms=1.6, zorder=3)
    M.basemap(ax, ne_dir, labels=False)
    M.region_boxes(ax, color="0.2", lw=0.6)
    M.excl_circle(ax)
    M.receptor(ax, ms=9)
    ax.set_title(title, loc="left", fontsize=8.5)
    if labels[0]:
        ax.set_ylabel("Latitude (°N)")
    if labels[1]:
        ax.set_xlabel("Longitude (°E)")
    return pm


def _cmap():
    return plt.get_cmap("RdBu_r")


def fig8(res_dir, out_png, ne_dir, n_days=None, field="E", sig_field="PSCF",
         layout=(("season", "S"), ("season", "L"), ("wind", "S"), ("wind", "L"))):
    z, _, _ = _load(res_dir)
    norm = LogNorm(vmin=0.25, vmax=4.0)
    cmap = _cmap()
    fig, axes = plt.subplots(2, 2, figsize=(7.4, 6.9), sharex=True, sharey=True)
    dlab = {"season": "season-matched", "wind": "wind-matched", "unmatched": "unmatched"}
    for k, (ax, (d, c)) in enumerate(zip(axes.ravel(), layout)):
        pm = _map_panel(ax, z, f"{d}__{c}__{field}_raw", f"{d}__{c}__w", z[f"sig__{d}__{sig_field}__{c}"],
                        f"({'abcd'[k]}) {CLS_LAB[c]}, {dlab[d]}", ne_dir, norm, cmap,
                        labels=(k % 2 == 0, k >= 2))
    cax = fig.add_axes([0.25, 0.045, 0.5, 0.018])
    cb = fig.colorbar(pm, cax=cax, orientation="horizontal", extend="both")
    cb.set_ticks([0.25, 0.5, 1, 2, 4])
    cb.set_ticklabels(["0.25", "0.5", "1", "2", "4"])
    cb.set_label("Enrichment, observed / expected under random assignment within strata")
    fig.subplots_adjust(left=0.08, right=0.98, top=0.96, bottom=0.14, wspace=0.06, hspace=0.16)
    fig.savefig(out_png)
    plt.close(fig)


def make_all(res_dir, fig_dir, ne_dir, n_days=None):
    fig_dir = Path(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig7(res_dir, fig_dir / "Fig7_class_agreement_vs_null.png")
    fig8(res_dir, fig_dir / "Fig8_enrichment_maps.png", ne_dir, n_days=n_days)
    print("trajectory figures written to", fig_dir)


**`rev_tables.py`**: Tables 4, 5 and S1 as CSV.

In [ ]:
%%writefile /content/rev_pkg/rev_tables.py
"""Tables 4, 5 and S1 of the revised manuscript (CSV, same layout) from the rev_traj_run output."""
from pathlib import Path

import pandas as pd

REGS = ["Syrian Desert", "An Nafud", "Rub al Khali", "Mesopotamia / Fertile Crescent", "Ad-Dahna corridor",
        "Northeast Africa", "Eastern Arabia / Gulf", "Iranian plateau / Sistan", "All eight regions"]
DES = {"unmatched": "Unmatched", "season": "Season-matched", "wind": "Wind-matched"}


def pv(p):
    return "<=0.001" if p < 0.0015 else f"{p:.3f}"


def _reg(res_dir):
    return pd.read_csv(Path(res_dir) / "region_summary_with_nulls.csv")


def _get(r, d, fld, c, reg):
    return r[(r.design == d) & (r.field == fld) & (r["class"] == c) & (r.region == reg)].iloc[0]


def table4(res_dir):
    """Table 4: PSCF enrichment by region, season- and wind-matched, S and L: mean (p), with coverage."""
    r = _reg(res_dir)
    rows = []
    for reg in REGS:
        b, bl = _get(r, "wind", "PSCF", "S", reg), _get(r, "wind", "PSCF", "L", reg)
        nm = [_get(r, d, "PSCF", c, reg).null_mean for d in ("season", "wind") for c in ("S", "L")]
        row = {"Region": reg + (" *" if (b.excluded_frac > 0.15 and reg != "All eight regions") else ""),
               "Share (%)": f"{100 * b.endpoint_share:.1f}",
               "Cells (W = 1)": f"{int(b.cells_occupied)} ({int(b.cells_W1)})",
               "Days (S / L)": f"{int(b.class_days_crossing)} / {int(bl.class_days_crossing)}",
               "Null mean": f"{sum(nm) / 4:.2f}"}
        for d in ("season", "wind"):
            for c in ("S", "L"):
                x = _get(r, d, "PSCF", c, reg)
                row[f"{DES[d]} {c}"] = f"{x['mean']:.2f} ({pv(x.p_region)})"
        rows.append(row)
    return pd.DataFrame(rows)


def table5(res_dir):
    """Table 5: wind-quintile composition of the season-matched sample (days / trajectories)."""
    t = pd.read_csv(Path(res_dir) / "table5_counts_by_quintile.csv")
    rows = []
    for _, r in t.iterrows():
        rows.append({"Quintile": int(r.quintile), "Wind (m/s)": f"{r.wind_lo:.1f}-{r.wind_hi:.1f}",
                     "All": f"{int(r.days)} / {int(r.traj)}", "S": f"{int(r.S_days)} / {int(r.S_traj)}",
                     "L": f"{int(r.L_days)} / {int(r.L_traj)}", "Overlap": int(r.both_days),
                     "S only": int(r.Sonly_days), "L only": int(r.Lonly_days),
                     "p_b (S / L)": f"{r.p_S:.3f} / {r.p_L:.3f}",
                     "Background (S / L)": f"{int(r.nonS_traj)} / {int(r.nonL_traj)}"})
    rows.append({"Quintile": "Total", "Wind (m/s)": f"{t.wind_lo.min():.1f}-{t.wind_hi.max():.1f}",
                 "All": f"{int(t.days.sum())} / {int(t.traj.sum())}", "S": f"{int(t.S_days.sum())} / {int(t.S_traj.sum())}",
                 "L": f"{int(t.L_days.sum())} / {int(t.L_traj.sum())}", "Overlap": int(t.both_days.sum()),
                 "S only": int(t.Sonly_days.sum()), "L only": int(t.Lonly_days.sum()),
                 "p_b (S / L)": f"{t.S_traj.sum() / t.traj.sum():.3f} / {t.L_traj.sum() / t.traj.sum():.3f}",
                 "Background (S / L)": f"{int(t.nonS_traj.sum())} / {int(t.nonL_traj.sum())}"})
    return pd.DataFrame(rows)


def tableS1(res_dir):
    """Table S1: (a-d) regional mean (p) for PSCF, PSCF exclusive classes, CWT max. extinction, CWT dust-code hours;
    (e) between-class correlations, observed (null mean)."""
    r = _reg(res_dir)
    out = []
    for panel, fld, cS, cL in [("(a) PSCF", "PSCF", "S", "L"), ("(b) PSCF, exclusive classes", "PSCF", "Sonly", "Lonly"),
                               ("(c) CWT, daily maximum extinction", "CWT_daymax", "S", "L"),
                               ("(d) CWT, dust-code hours", "CWT_hours", "S", "L")]:
        if r[(r.field == fld)].empty:
            continue
        for reg in REGS:
            row = {"Panel": panel, "Region": reg}
            for d in DES:
                for lab, c in (("S", cS), ("L", cL)):
                    x = _get(r, d, fld, c, reg)
                    row[f"{DES[d]} {lab}"] = f"{x['mean']:.2f} ({pv(x.p_region)})"
            out.append(row)
    nul = pd.read_csv(Path(res_dir) / "permutation_null_correlations.csv")
    corr = []
    for fld in ["PSCF", "CWT_daymax", "CWT_hours"]:
        for pair, plab in [("L-S", "overlapping"), ("Lonly-Sonly", "exclusive")]:
            row = {"Panel": "(e) Between-class correlation, observed (null mean)", "Region": f"{fld}, {plab}"}
            for d in DES:
                x = nul[(nul.design == d) & (nul.field == fld) & (nul.pair == pair)]
                if x.empty:
                    continue
                x = x.iloc[0]
                row[f"{DES[d]} weighted"] = f"{x.r_obs_weighted:.2f} ({x.null_w_mean:.2f})"
                row[f"{DES[d]} unweighted"] = f"{x.r_obs_unweighted:.2f} ({x.null_u_mean:.2f})"
            corr.append(row)
    return pd.DataFrame(out), pd.DataFrame(corr)


def make_all(res_dir, out_dir=None):
    out = Path(out_dir or res_dir)
    out.mkdir(parents=True, exist_ok=True)
    t4, t5 = table4(res_dir), table5(res_dir)
    s1, s1e = tableS1(res_dir)
    t4.to_csv(out / "Table4_regions.csv", index=False)
    t5.to_csv(out / "Table5_counts_by_wind_quintile.csv", index=False)
    s1.to_csv(out / "TableS1_regions_all_designs.csv", index=False)
    s1e.to_csv(out / "TableS1e_correlations.csv", index=False)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    print(t4.to_string(index=False))
    print()
    print(t5.to_string(index=False))
    return t4, t5


**`rev_upwind.py`**: OPTIONAL upwind ERA5 diagnostics (not used in the revised manuscript).

In [ ]:
%%writefile /content/rev_pkg/rev_upwind.py
"""OPTIONAL: ERA5 monthly means over the upwind source boxes (2001-2025).

Not needed for the revision. The revised manuscript treats upwind mechanisms as hypotheses
and names the data that would test them; a full upwind attribution is the planned follow-up
study. Running this cell gives a first look (soil moisture, precipitation and 10 m wind over
the eight source boxes), for the authors' information.
"""
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import paths as P
import rev_maps as M
from figstyle import shade_min

BOXES = dict(M.REGIONS)
BOXES["Riyadh receptor"] = (24.5, 25.25, 46.5, 47.0)


def retrieve(cds_key):
    raw = P.REV / "ERA5_monthly_upwind_raw.nc"
    if not raw.exists():
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "ecmwf-datastores-client"], check=True)
        from ecmwf.datastores import Client
        client = Client(url="https://cds.climate.copernicus.eu/api", key=cds_key)
        client.retrieve("reanalysis-era5-single-levels-monthly-means", {
            "product_type": ["monthly_averaged_reanalysis"],
            "variable": ["volumetric_soil_water_layer_1", "total_precipitation", "10m_wind_speed", "land_sea_mask"],
            "year": [str(y) for y in range(2001, 2026)], "month": [f"{m:02d}" for m in range(1, 13)],
            "time": ["00:00"], "area": [38, 25, 15, 65],
            "data_format": "netcdf", "download_format": "unarchived"}, target=str(raw))
    return raw


def _open_any(path):
    import xarray as xr
    if open(path, "rb").read(4) == b"PK\x03\x04":
        tmp = Path("/content/era5_upwind") if Path("/content").exists() else P.REV / "era5_upwind"
        tmp.mkdir(exist_ok=True)
        with zipfile.ZipFile(path) as z:
            z.extractall(tmp)
        ds = xr.merge([xr.open_dataset(p) for p in sorted(tmp.glob("*.nc"))], compat="override", join="outer")
    else:
        ds = xr.open_dataset(path)
    if "expver" in ds.dims:
        ds = ds.max("expver")
    elif "expver" in ds.variables:
        ds = ds.drop_vars("expver")
    return ds.rename({"valid_time": "time"}) if "valid_time" in ds.dims else ds


def box_means(raw):
    import xarray as xr
    ds = _open_any(raw)
    lsm = ds["lsm"].isel(time=0) if "time" in ds["lsm"].dims else ds["lsm"]
    rows = []
    for name, (la0, la1, lo0, lo1) in BOXES.items():
        ii = np.where((ds.latitude >= la0) & (ds.latitude <= la1))[0]
        jj = np.where((ds.longitude >= lo0) & (ds.longitude <= lo1))[0]
        sub = ds.isel(latitude=ii, longitude=jj)
        land = lsm.isel(latitude=ii, longitude=jj) >= 0.5
        w = np.cos(np.deg2rad(sub.latitude)) * xr.ones_like(sub.longitude)
        out = {}
        for v, ok in [("swvl1", land), ("tp", xr.ones_like(land, dtype=bool)), ("si10", land)]:
            if v in sub:
                ww = w.where(ok)
                out[v] = (sub[v] * ww).sum(("latitude", "longitude")) / ww.sum()
        df = pd.DataFrame({k: s.values for k, s in out.items()}, index=pd.to_datetime(sub.time.values))
        df["region"] = name
        rows.append(df)
    res = pd.concat(rows).rename_axis("time").reset_index()
    res["tp_mm_day"] = res.pop("tp") * 1000.0
    res.to_csv(P.UPWIND_CSV, index=False)
    return res


def figure(res=None, regions=("Mesopotamia / Fertile Crescent", "Syrian Desert", "An Nafud", "Riyadh receptor")):
    res = pd.read_csv(P.UPWIND_CSV, parse_dates=["time"]) if res is None else res
    res["year"] = res.time.dt.year
    res["month"] = res.time.dt.month
    # hydrological year ending in the dust season: precipitation Oct(y-1)-May(y); soil moisture and wind Mar-May(y)
    res["hy"] = np.where(res.month >= 10, res.year + 1, res.year)
    wet = res[res.month.isin([10, 11, 12, 1, 2, 3, 4, 5])].groupby(["region", "hy"]).tp_mm_day.mean().rename("P_OctMay")
    mam = res[res.month.isin([3, 4, 5])].groupby(["region", "year"])[["swvl1", "si10"]].mean()
    mam.index.names = ["region", "hy"]
    t = pd.concat([wet, mam], axis=1).reset_index().rename(columns={"hy": "year"})
    t = t[t.year.between(2002, 2025)]
    t.to_csv(P.OUT / "upwind_dust_season_means.csv", index=False)
    fig, axes = plt.subplots(3, 1, figsize=(7.0, 7.5), sharex=True)
    specs = [("P_OctMay", "Precipitation Oct–May\n(mm day$^{-1}$)"), ("swvl1", "Soil moisture MAM\n(m$^3$ m$^{-3}$)"),
             ("si10", "10 m wind MAM\n(m s$^{-1}$)")]
    for ax, (v, lab) in zip(axes, specs):
        for reg in regions:
            s = t[t.region == reg]
            base = s[s.year.between(2016, 2022)][v].mean()
            ax.plot(s.year, s[v] / base, "o-", ms=3, lw=1.2, label=reg)
        ax.axhline(1, color="0.5", lw=0.7)
        ax.set_ylabel(lab + "\nrelative to 2016–2022")
        shade_min(ax)
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=7.5)
    axes[-1].set_xlabel("Year")
    fig.tight_layout()
    fig.savefig(P.FIGS / "optional_upwind_era5.png")
    plt.close(fig)
    return t


## 2. Export trajectory statistics (required once)
Re-reads the 14,612 `tdump` files in `05_HYSPLIT/trajectories` (or the cached endpoint parquet if it exists) and saves per-trajectory endpoint counts on the 1° grid after the 3 h and 300 km filters. Nothing is recomputed and no existing file is changed. Skips itself if the file already exists.

In [ ]:
# 2.1
import rev_export
rev_export.export_traj()

## 3. Compact METAR export (optional)
Enables the dust-hour visibility intensity (±3 h of trajectory arrival) as a third CWT intensity. Seconds.

In [ ]:
# 3.1
import rev_export
rev_export.export_metar()

## 4. Catalogue statistics (optional; values already in the manuscript)
Class counts with overlap and exclusive classes, Table 1 with bootstrap intervals and the rank test, year-by-year prevalence with Wilson intervals (reference only), and the hourly reconciliation (41 unassigned reports, visibility ceiling; needs the METAR file).

In [ ]:
# 4.1
import json, rev_stats as S
d = S.load_class()
print(json.dumps(S.class_counts(d), indent=1))
t, ch, ranks, p = S.table1(d)
print(ch.round(1).to_string(index=False))
print("three lowest years:", ranks, " rank-test p =", round(p, 4))
print(S.prevalence(d).round(1).to_string(index=False))
print(S.hourly_checks())

## 5. Calibrated expectation and benchmarks (optional; ~1 min)
Leave-one-year-out random-forest scores, Platt calibration, block-bootstrap prediction intervals with the dispersion factor, constant and seasonal climatology. Writes `calib_yearly.csv` (Table 3, Figure 6) and `calib_skill.csv` (Table 2 benchmark rows).

In [ ]:
# 5.1
%run /content/rev_pkg/rev_calib.py

## 6. Figures 1-6 (optional)

In [ ]:
# 6.1  (run 5.1 first: Figure 6 reads calib_yearly.csv)
import rev_figs
rev_figs.make_all()

## 7. Trajectory re-analysis (required)
Three background designs (unmatched, season-matched, wind-matched) × four classes (overlapping and mutually exclusive) × PSCF and class-conditional CWT, with day-block permutation nulls (B = 1000) inside the strata of each design. Writes Tables 4, 5 and S1 (CSV) and Figures 7-8. Also reproduces the submitted wind-matched numbers (`pooled_vs_standardised_regions.csv`) so the changes can be traced.

In [ ]:
# 7.1
import rev_traj_run as RR, rev_traj_figs as TF, rev_tables as TT, paths as P
TRAJ_OUT = P.OUT / "trajectory"
metar = P.METAR_COMPACT if P.METAR_COMPACT.exists() else None
res = RR.run_all(P.NPZ, P.CLASS_CSV, TRAJ_OUT, metar=metar, B=1000)
TT.make_all(TRAJ_OUT)
n_days = {c: res["design_sizes"]["wind"][f"{c}_days"] for c in ["S", "L", "Sonly", "Lonly"]}
TF.make_all(TRAJ_OUT, P.FIGS, P.NE_DIR, n_days=n_days)

## 8. Upwind ERA5 diagnostics (optional, not used in the manuscript)
Monthly ERA5 soil moisture (0-7 cm), precipitation and 10 m wind averaged over the eight source boxes, 2001-2025. The revised manuscript deliberately treats upwind mechanisms as hypotheses and leaves their test to the planned follow-up study; use this only for your own information. Needs your CDS API key (entered interactively, never stored).

In [ ]:
# 8.1  Retrieve (one request; the CDS queue can take 10-40 min)
import getpass, rev_upwind as U
raw = U.retrieve(getpass.getpass("CDS API key: ").strip())

In [ ]:
# 8.2  Box means and figure (optional_upwind_era5.png)
res_up = U.box_means(raw)
U.figure(res_up).round(3).tail(12)

## 9. Collect outputs

In [ ]:
# 9.1  Zip the CSV outputs and figures (they are also on Drive under 07_Revision/)
import zipfile, paths as P
zip_path = "/content/revision_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in (P.OUT, P.FIGS):
        for f in sub.rglob("*"):
            if f.is_file() and f.suffix in {".csv", ".png", ".json"}:
                z.write(f, f.relative_to(P.REV))
print(zip_path)
from google.colab import files
files.download(zip_path)